# 800 — External Bulk Cohorts

## PREFLIGHT-A — CPTAC-3 cohort eligibility and frozen-input readiness

**Status:** implementation draft; no biological replication analysis has been authorized or executed.

**Objective.** Convert the Notebook 111 CPTAC-3 acquisition audit into auditable, *technical* candidate inventories for prospective external-bulk validation of the three frozen Phase 4 consensus transcriptomic representations.

This initial Notebook 800 implementation is restricted to: (1) referencing the frozen acquisition evidence; (2) auditing primary-tumor candidate coverage and case multiplicity; (3) checking *lexical* overlap against the frozen TCGA sample mapping; (4) inspecting a deterministic, technical STAR Counts gene-ID pilot; and (5) writing non-frozen preflight diagnostics. It does **not** project consensus scores, construct a final analysis cohort, assess cross-dataset replication, test program concordance, or interpret methylation–expression evidence.

**Scientific boundaries.** Cohort eligibility, lineage crosswalk, donor independence, a one-record-per-case rule, full gene coverage, normalization, the replication estimand, null models and multiplicity handling must be prospectively reviewed and frozen before those analyses. Cohort source labels (`primary_site`, `disease_type`) must not be silently converted into TCGA lineages. CPTAC-3 RNA-seq eligibility must not be conditioned on DNA-methylation availability.

**Sources.** `111_cptac3_acquisition_and_audit.ipynb`, `docs/contracts/phase8/NOTEBOOK_800_PREFLIGHT_A_DRAFT.md`, `docs/NOTEBOOK_STYLE_GUIDE.md`, `config/artifact_registry.json` and `config/raw_data_registry.json`.

Run from the project's **Python 3.11.8** Jupyter environment with the local Notebook 111 outputs already generated. No GDC calls, re-download, upstream re-execution or full raw payload MD5 scan is performed.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json
import platform

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.artifact_registry import (
    get_artifact,
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path
from pancancer_epigenetics.utils.raw_data_registry import load_raw_data_registry

In [2]:
# =============================================================================
# Notebook 111 handoff and stable Phase 4 input identifiers
# =============================================================================

AUDIT_SUMMARY_PATH = Paths.metadata / "111_cptac3_acquisition_audit_summary.json"
RAW_REGISTRY_PROPOSAL_PATH = Paths.metadata / "111_cptac3_raw_registry_proposal.json"
OUTPUT_DIR = Paths.metadata

AUDIT_INPUT_KEYS = (
    "file_index",
    "candidate_primary",
    "ambiguity_by_lineage",
)

FROZEN_ARTIFACT_IDS = {
    "gene_universe": "phase4.400.cross_system_shared_gene_universe",
    "program_catalog": "phase4.401.consensus_transcriptomic_program_catalog",
    "program_weights": "phase4.401.consensus_transcriptomic_gene_weights",
    "construction_metadata": "phase4.401.consensus_construction_metadata",
    "tcga_sample_mapping": "phase2.203.final_sample_mapping",
}

OUTPUT_PATHS = {
    "lineage_inventory": OUTPUT_DIR / "800_preflight_a_candidate_lineage_inventory.csv",
    "case_multiplicity": OUTPUT_DIR / "800_preflight_a_case_multiplicity.csv",
    "identifier_collisions": OUTPUT_DIR / "800_preflight_a_identifier_collisions.csv",
    "gene_id_pilot": OUTPUT_DIR / "800_preflight_a_gene_id_pilot.csv",
    "readiness": OUTPUT_DIR / "800_preflight_a_readiness.json",
    "rna_adjudication_case_evidence": OUTPUT_DIR / "800_preflight_b_rna_case_evidence.csv",
    "rna_adjudication_file_evidence": OUTPUT_DIR / "800_preflight_b_rna_file_evidence.csv",
    "rna_c_field_coverage": OUTPUT_DIR / "800_preflight_c_rna_field_coverage.csv",
    "rna_c_file_availability": OUTPUT_DIR / "800_preflight_c_rna_file_availability.csv",
    "rna_c_case_coverage": OUTPUT_DIR / "800_preflight_c_rna_case_coverage.csv",
    "rna_d_live_gdc_snapshot": OUTPUT_DIR / "800_preflight_d_gdc_targeted_pilot.json",
    "rna_d_live_gdc_snapshot_v2": OUTPUT_DIR / "800_preflight_d_gdc_targeted_pilot_v2.json",
    "rna_d_identity": OUTPUT_DIR / "800_preflight_d_rna_pilot_file_identity.csv",
    "rna_d_field_evidence": OUTPUT_DIR / "800_preflight_d_rna_pilot_field_evidence.csv",
    "rna_d_case_summary": OUTPUT_DIR / "800_preflight_d_rna_pilot_case_summary.csv",
    "rna_d_case_field_comparison": OUTPUT_DIR / "800_preflight_d_rna_pilot_case_field_comparison.csv",
    "rna_e_file_diagnostics": OUTPUT_DIR / "800_preflight_e_rna_single_primary_file_technical.csv",
    "rna_e_gene_coverage": OUTPUT_DIR / "800_preflight_e_frozen_gene_id_coverage.csv",
    "rna_e_source_attrition": OUTPUT_DIR / "800_preflight_e_source_context_case_attrition.csv",
}

In [3]:
# =============================================================================
# Read Notebook 111's observed audit boundary
# =============================================================================

with AUDIT_SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    audit_summary = json.load(handle)

if audit_summary.get("notebook") != 111:
    raise ValueError("The source is not a Notebook 111 audit summary")
if audit_summary.get("cohort_frozen") is not False:
    raise ValueError("Unexpected cohort-freeze flag in Notebook 111 handoff")
if audit_summary.get("phase8_validation_performed") is not False:
    raise ValueError("Unexpected Phase 8 validation status")
if audit_summary.get("n_files_by_modality") != {
    "rnaseq": 2968, "methylation": 2610,
}:
    raise ValueError("CPTAC-3 source-snapshot identity differs from Notebook 111")

print("Notebook 111 source audit: PASS (reported)")
print("Cohort freeze:", audit_summary["cohort_frozen"])
print("External validation performed:", audit_summary["phase8_validation_performed"])

Notebook 111 source audit: PASS (reported)
Cohort freeze: False
External validation performed: False


In [4]:
# =============================================================================
# Resolve and fingerprint the small derived audit tables consumed by 800
# =============================================================================

audit_inputs = {}
for name in AUDIT_INPUT_KEYS:
    record = audit_summary["audit_artifacts"][name]
    path = Paths.root / record["path"]
    if not path.is_file():
        raise FileNotFoundError(f"Missing Notebook 111 derived handoff: {path}")
    if calculate_sha256(path) != record["sha256"]:
        raise ValueError(f"Notebook 111 derived handoff changed: {name}")
    audit_inputs[name] = path

print("Verified Notebook 111 derived handoffs:")
for name, path in audit_inputs.items():
    print(" ", name, project_relative_path(path))

Verified Notebook 111 derived handoffs:
  file_index data/interim/metadata/111_cptac3_file_index.csv
  candidate_primary data/interim/metadata/111_cptac3_provisional_primary_tumor_files.csv
  ambiguity_by_lineage data/interim/metadata/111_cptac3_ambiguity_by_lineage.csv


In [5]:
# =============================================================================
# Read authoritative Notebook 111 tables for new downstream questions only
# =============================================================================

file_index = pd.read_csv(audit_inputs["file_index"], dtype="string")
candidate_primary = pd.read_csv(audit_inputs["candidate_primary"], dtype="string")
upstream_ambiguity = pd.read_csv(
    audit_inputs["ambiguity_by_lineage"], dtype="string"
)

new_use_columns = {
    "modality", "file_id", "filename", "case_uuid", "case_submitter_id",
    "sample_uuid", "aliquot_uuid", "mapping_status",
    "primary_site", "disease_type", "platform",
}
missing_columns = new_use_columns - set(file_index.columns)
if missing_columns:
    raise KeyError(f"Notebook 800 required columns missing: {sorted(missing_columns)}")

# Notebook 111's mapping-status distribution, source totals and primary
# sample-type inventory are authoritative and are not recalculated here.

## Source-registration boundary

The Notebook 111 registry proposal is a review artifact, **not** permission to silently modify the raw-data registry. A missing `cptac3` entry must remain visible as a registration task. Likewise, Phase 4 inputs are frozen and consumed only after exact identity checks; their weights are *not* applied to CPTAC-3 in this preflight.

In [6]:
# =============================================================================
# Inspect CPTAC-3 raw registry state without mutating it
# =============================================================================

raw_registry = load_raw_data_registry()
with RAW_REGISTRY_PROPOSAL_PATH.open("r", encoding="utf-8") as handle:
    cptac3_registry_proposal = json.load(handle)

cptac3_registered = "cptac3" in raw_registry
print("CPTAC-3 registered:", cptac3_registered)
print("Proposed RNA status:",
      cptac3_registry_proposal["cohorts"]["rnaseq"]["status"])
print("Proposed methylation status:",
      cptac3_registry_proposal["cohorts"]["methylation"]["status"])
if cptac3_registered:
    print("Existing registration requires review against Notebook 111 proposal")

CPTAC-3 registered: False
Proposed RNA status: acquired_not_used
Proposed methylation status: acquired_not_used


In [7]:
# =============================================================================
# Verify exact frozen Phase 4 and TCGA reference identities
# =============================================================================

artifact_registry = load_artifact_registry()
frozen_paths = {}
for name, artifact_id in FROZEN_ARTIFACT_IDS.items():
    spec = get_artifact(artifact_registry, artifact_id)
    path = resolve_artifact_path(artifact_registry, artifact_id)
    if spec["status"] != "frozen":
        raise ValueError(f"Required upstream artifact is not frozen: {artifact_id}")
    if path.stat().st_size != spec["size_bytes"]:
        raise ValueError(f"Frozen upstream file size changed: {artifact_id}")
    if calculate_sha256(path) != spec["sha256"]:
        raise ValueError(f"Frozen upstream fingerprint changed: {artifact_id}")
    frozen_paths[name] = path

print("Verified", len(frozen_paths), "frozen input identities")

Verified 5 frozen input identities


In [8]:
# =============================================================================
# Read the frozen gene universe and consensus metadata without scoring
# =============================================================================

shared_genes = pd.read_csv(frozen_paths["gene_universe"], dtype="string")
program_catalog = pd.read_csv(frozen_paths["program_catalog"], dtype="string")
program_weights = pd.read_csv(frozen_paths["program_weights"], dtype="string")
with frozen_paths["construction_metadata"].open("r", encoding="utf-8") as handle:
    construction_metadata = json.load(handle)

required_gene_columns = {"gene_symbol", "tumor_gene_id", "cell_line_gene_id"}
if required_gene_columns - set(shared_genes.columns):
    raise KeyError("Frozen gene universe lacks identifiers required for a technical crosswalk")
if {"consensus_program_id", "gene_symbol", "consensus_weight"} - set(program_weights):
    raise KeyError("Frozen gene weights lack identity columns")

print("Frozen genes:", len(shared_genes))
print("Program IDs:", sorted(program_weights["consensus_program_id"].unique().tolist()))
print("Frozen weight rows:", len(program_weights))
print("Score standardization:", construction_metadata["score_construction"])

Frozen genes: 2389
Program IDs: ['CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03']
Frozen weight rows: 7167
Score standardization: {'gene_standardization': 'within_system_mean0_sd1', 'projection': 'standardized_expression_dot_consensus_loading', 'final_score_standardization': 'within_system_per_program_mean0_sd1', 'cross_system_joint_normalization': False, 'absolute_score_scale_cross_system_comparable': False}


## Lineage-aware review of **new** case-level eligibility risks

Notebook 111 already froze technical evidence on sample ancestry, including an
`ambiguity_by_lineage` table with **file-level** denominators and a provisional
primary-tumor inventory. Notebook 800 consumes those audited numbers directly.

The new question is case-level: how many *distinct cases* in each existing
GDC stratum are touched by unresolved file ancestry, and how does that compare
descriptively with the pool of uniquely mapped provisional primary-tumor cases?
These are **not** new primary-tumor calls, new lineage mappings, or eligibility
criteria. The denominator of the unresolved-case fraction contains **all sample
types** from the acquired files, and cannot be interpreted as primary-tumor
attrition. A dedicated primary-compatible unresolved-sample sensitivity, and
the final one-case selection rule, remain separate decisions.

In [9]:
# =============================================================================
# Reuse Notebook 111 source-stratum counts; compute only new case-level risk
# =============================================================================

STRATUM_KEYS = ["modality", "primary_site", "disease_type", "platform"]
UPSTREAM_COUNT_COLUMNS = [
    "n_files", "n_cases", "n_ambiguous_files", "ambiguous_file_fraction",
]
missing_upstream = (
    set(STRATUM_KEYS + UPSTREAM_COUNT_COLUMNS) - set(upstream_ambiguity.columns)
)
if missing_upstream:
    raise KeyError(f"111 lineage audit missing fields: {sorted(missing_upstream)}")

source_counts = upstream_ambiguity[
    STRATUM_KEYS + UPSTREAM_COUNT_COLUMNS
].copy()
for column in UPSTREAM_COUNT_COLUMNS:
    source_counts[column] = pd.to_numeric(
        source_counts[column], errors="raise"
    )

unresolved_case_counts = (
    file_index.loc[file_index["mapping_status"].eq("ambiguous_sample_ancestry")]
    .groupby(STRATUM_KEYS, dropna=False)["case_uuid"]
    .nunique()
    .rename("cases_with_ambiguous_files")
    .reset_index()
)
source_counts = source_counts.merge(
    unresolved_case_counts, on=STRATUM_KEYS, how="left", validate="one_to_one"
)
source_counts["cases_with_ambiguous_files"] = (
    source_counts["cases_with_ambiguous_files"].fillna(0).astype("int64")
)
source_counts["fraction_all_cases_with_ambiguous_files"] = (
    source_counts["cases_with_ambiguous_files"] / source_counts["n_cases"]
)

In [10]:
# =============================================================================
# Contextualize new case-level ambiguity against the 111 primary inventory
# =============================================================================

# Notebook 111 reports primary counts by source site/disease; this new
# platform-specific case view is descriptive only and is not an eligibility gate.
primary_case_counts = (
    candidate_primary.groupby(STRATUM_KEYS, dropna=False)["case_uuid"]
    .nunique()
    .rename("provisional_primary_cases")
    .reset_index()
)

lineage_inventory = source_counts.merge(
    primary_case_counts, on=STRATUM_KEYS, how="left", validate="one_to_one"
)
lineage_inventory["provisional_primary_cases"] = (
    lineage_inventory["provisional_primary_cases"].fillna(0).astype("int64")
)
lineage_inventory = lineage_inventory.sort_values(
    ["modality", "cases_with_ambiguous_files", "primary_site", "disease_type"],
    ascending=[True, False, True, True], kind="stable"
).reset_index(drop=True)

display(lineage_inventory[
    ["modality", "primary_site", "disease_type", "platform",
     "n_cases", "cases_with_ambiguous_files",
     "fraction_all_cases_with_ambiguous_files", "provisional_primary_cases"]
].head(25))
print("Lineage crosswalk and primary-specific attrition: NOT ASSESSED")

,modality,primary_site,disease_type,platform,n_cases,cases_with_ambiguous_files,fraction_all_cases_with_ambiguous_files,provisional_primary_cases
0,methylation,Pancreas,Ductal and Lobular Neoplasms,Illumina Methylation Epic,146,62,0.424658,83
1,methylation,Bronchus and lung,Adenomas and Adenocarcinomas,Illumina Methylation Epic,230,58,0.252174,177
2,methylation,Brain,Gliomas,Illumina Methylation Epic,205,38,0.185366,161
3,methylation,Kidney,Adenomas and Adenocarcinomas,Illumina Methylation Epic,256,27,0.105469,231
4,methylation,Skin,Nevi and Melanomas,Illumina Methylation Epic,159,24,0.150943,134
5,methylation,Bronchus and lung,Squamous Cell Neoplasms,Illumina Methylation Epic,109,21,0.192661,89
6,methylation,"Uterus, NOS",Adenomas and Adenocarcinomas,Illumina Methylation Epic,241,20,0.082988,220
7,methylation,Larynx,Squamous Cell Neoplasms,Illumina Methylation Epic,49,15,0.306122,35
8,methylation,Stomach,Adenomas and Adenocarcinomas,Illumina Methylation Epic v2,164,11,0.067073,151
9,methylation,Skin,Nevi and Melanomas,Illumina Methylation Epic v2,18,6,0.333333,1


Lineage crosswalk and primary-specific attrition: NOT ASSESSED


In [11]:
# =============================================================================
# Quantify case-level multiplicity in provisional primary records
# =============================================================================

case_multiplicity = (
    candidate_primary.groupby(["modality", "case_uuid"], dropna=False)
    .agg(
        n_files=("file_id", "nunique"),
        n_samples=("sample_uuid", "nunique"),
        n_aliquots=("aliquot_uuid", "nunique"),
        n_platforms=("platform", "nunique"),
        file_uuid_values=("file_id", lambda v: "|".join(sorted(set(v.dropna())))),
        sample_uuid_values=("sample_uuid", lambda v: "|".join(sorted(set(v.dropna())))),
        aliquot_uuid_values=("aliquot_uuid", lambda v: "|".join(sorted(set(v.dropna())))),
        primary_site_values=("primary_site", lambda v: "|".join(sorted(set(v.dropna())))),
        disease_type_values=("disease_type", lambda v: "|".join(sorted(set(v.dropna())))),
    )
    .reset_index()
)

case_multiplicity["needs_selection_policy"] = (
    case_multiplicity[["n_files", "n_samples", "n_aliquots"]].gt(1).any(axis=1)
)
case_multiplicity["selected_file_id"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="string"
)
case_multiplicity["selection_status"] = "NOT_FROZEN"
case_multiplicity = case_multiplicity.sort_values(
    ["modality", "case_uuid"], kind="stable"
).reset_index(drop=True)

display(
    case_multiplicity.groupby("modality")
    .agg(
        candidate_cases=("case_uuid", "nunique"),
        cases_needing_review=("needs_selection_policy", "sum"),
        maximum_files_per_case=("n_files", "max"),
        maximum_samples_per_case=("n_samples", "max"),
    )
)
print("No case-level selection, averaging or technical-replicate pooling performed")

,candidate_cases,cases_needing_review,maximum_files_per_case,maximum_samples_per_case
modality,,,,
methylation,1336,62,5,5
rnaseq,1370,288,8,5


No case-level selection, averaging or technical-replicate pooling performed


## RNA-seq per-case multiplicity: outcome-blind structural classification

The initial 800 preflight identified **288 provisional RNA-seq cases requiring multiplicity review**. The next technical task is to distinguish case-level identifier structures **without** reopening Notebook 111's file-to-sample ancestry audit or making a biological sample-selection decision.

Classification is based only on already audited file, sample, and aliquot UUIDs. **More than one file on a single aliquot is not, by itself, proof of technical replication or equivalence of processed files.** More than one aliquot under one sample is not automatically exchangeable, and distinct sample UUIDs under one case cannot be pooled as independent patients.

Cases with internally inconsistent counts or cross-file mapping conflicts must remain explicitly unresolved. Even a case with one file is merely *technically simple*, **not** finally eligible: donor independence, lineage harmonization, technical QC and prospective Phase 8 scoring policy remain open.

In [12]:
# =============================================================================
# Classify RNA-seq case multiplicity from the existing 800 case inventory
# =============================================================================

rna_case_mask = case_multiplicity["modality"].eq("rnaseq")
rna_cases = case_multiplicity.loc[rna_case_mask].copy()

inconsistent_counts = (
    rna_cases[["n_files", "n_samples", "n_aliquots"]].le(0).any(axis=1)
    | rna_cases["n_aliquots"].gt(rna_cases["n_files"])
    | rna_cases["n_samples"].gt(rna_cases["n_aliquots"])
)
rna_cases["rna_multiplicity_class"] = np.select(
    [
        inconsistent_counts,
        rna_cases["n_samples"].gt(1),
        rna_cases["n_aliquots"].gt(1),
        rna_cases["n_files"].gt(1),
    ],
    [
        "INCONSISTENT_BIOSPECIMEN_COUNTS",
        "MULTIPLE_SAMPLES_IN_CASE",
        "ONE_SAMPLE_MULTIPLE_ALIQUOTS",
        "ONE_ALIQUOT_MULTIPLE_FILES",
    ],
    default="SINGLE_FILE_SAMPLE_ALIQUOT",
)
print("RNA-seq provisional cases classified:", len(rna_cases))
print("No record selected, pooled, excluded or scored")

RNA-seq provisional cases classified: 1370
No record selected, pooled, excluded or scored


In [13]:
# =============================================================================
# Screen for cross-file ancestry conflicts not resolved by single-file links
# =============================================================================

rna_candidate_rows = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq"),
    ["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"],
].copy()

aliquot_links = (
    rna_candidate_rows.groupby("aliquot_uuid")
    .agg(
        n_parent_samples=("sample_uuid", "nunique"),
        n_parent_cases=("case_uuid", "nunique"),
    )
)
sample_links = (
    rna_candidate_rows.groupby("sample_uuid")
    .agg(n_parent_cases=("case_uuid", "nunique"))
)
conflicting_aliquots = set(aliquot_links.loc[
    aliquot_links["n_parent_samples"].gt(1)
    | aliquot_links["n_parent_cases"].gt(1)
].index)
conflicting_samples = set(sample_links.loc[
    sample_links["n_parent_cases"].gt(1)
].index)

rna_cases["cross_file_ancestry_conflict"] = (
    rna_cases["aliquot_uuid_values"].map(
        lambda s: any(part in conflicting_aliquots for part in s.split("|"))
    )
    | rna_cases["sample_uuid_values"].map(
        lambda s: any(part in conflicting_samples for part in s.split("|"))
    )
)
rna_cases.loc[
    rna_cases["cross_file_ancestry_conflict"],
    "rna_multiplicity_class",
] = "CROSS_FILE_BIOSPECIMEN_CONFLICT"

print("Aliquot UUIDs with incompatible cross-file parents:", len(conflicting_aliquots))
print("Sample UUIDs with incompatible cross-file cases:", len(conflicting_samples))
print("Affected RNA cases:", int(rna_cases["cross_file_ancestry_conflict"].sum()))

Aliquot UUIDs with incompatible cross-file parents: 0
Sample UUIDs with incompatible cross-file cases: 0
Affected RNA cases: 0


In [14]:
# =============================================================================
# Show nonoverlapping RNA-seq structural classes and review candidates
# =============================================================================

rna_class_summary = (
    rna_cases.groupby("rna_multiplicity_class", dropna=False)
    .agg(
        n_cases=("case_uuid", "nunique"),
        max_files_in_case=("n_files", "max"),
        max_samples_in_case=("n_samples", "max"),
    )
    .reset_index()
    .sort_values(["n_cases", "rna_multiplicity_class"], ascending=[False, True])
)
rna_class_summary["fraction_provisional_rna_cases"] = (
    rna_class_summary["n_cases"] / len(rna_cases)
)

display(rna_class_summary)
print(
    "Cases previously flagged for multiplicity review:",
    int(rna_cases["needs_selection_policy"].sum()),
)
print("Unresolved case selection: ALL — no file selection rule frozen")

# Preserve exact source labels; mixed labels must remain visible.
display(
    rna_cases.loc[rna_cases["needs_selection_policy"]]
    .groupby(["primary_site_values", "rna_multiplicity_class"], dropna=False)
    .size().rename("n_cases").reset_index()
    .sort_values(["n_cases", "primary_site_values"], ascending=[False, True])
    .head(20)
)

,rna_multiplicity_class,n_cases,max_files_in_case,max_samples_in_case,fraction_provisional_rna_cases
2,SINGLE_FILE_SAMPLE_ALIQUOT,1082,1,1,0.789781
0,MULTIPLE_SAMPLES_IN_CASE,276,8,5,0.201460
1,ONE_SAMPLE_MULTIPLE_ALIQUOTS,12,3,1,0.008759


Cases previously flagged for multiplicity review: 288
Unresolved case selection: ALL — no file selection rule frozen


,primary_site_values,rna_multiplicity_class,n_cases
13,"Uterus, NOS",MULTIPLE_SAMPLES_IN_CASE,179
4,Kidney,MULTIPLE_SAMPLES_IN_CASE,40
5,Larynx,MULTIPLE_SAMPLES_IN_CASE,19
11,Other and unspecified parts of tongue,MULTIPLE_SAMPLES_IN_CASE,12
12,Pancreas,ONE_SAMPLE_MULTIPLE_ALIQUOTS,12
2,Floor of mouth,MULTIPLE_SAMPLES_IN_CASE,11
3,Gum,MULTIPLE_SAMPLES_IN_CASE,3
7,Oropharynx,MULTIPLE_SAMPLES_IN_CASE,3
9,"Other and ill-defined sites in lip, oral cavit...",MULTIPLE_SAMPLES_IN_CASE,3
6,Lip,MULTIPLE_SAMPLES_IN_CASE,2


In [15]:
# =============================================================================
# Annotate the existing case-level output; create no duplicate source inventory
# =============================================================================

case_multiplicity["rna_multiplicity_class"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="string"
)
case_multiplicity["rna_cross_file_ancestry_conflict"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="boolean"
)

case_multiplicity.loc[
    rna_case_mask, "rna_multiplicity_class"
] = rna_cases["rna_multiplicity_class"].to_numpy()
case_multiplicity.loc[
    rna_case_mask, "rna_cross_file_ancestry_conflict"
] = rna_cases["cross_file_ancestry_conflict"].to_numpy()

if (
    rna_class_summary["n_cases"].sum() != len(rna_cases)
    or case_multiplicity.loc[rna_case_mask, "rna_multiplicity_class"].isna().any()
):
    raise ValueError("Incomplete RNA-seq case multiplicity classification")

print("Annotated output:", project_relative_path(OUTPUT_PATHS["case_multiplicity"]))
print("Classification: descriptive only; no sample selected")

Annotated output: data/interim/metadata/800_preflight_a_case_multiplicity.csv
Classification: descriptive only; no sample selected


## PREFLIGHT-B — Outcome-blind RNA-seq multiplicity adjudication evidence

**Question:** for the 288 cases containing multiple uniquely mapped provisional primary-tumor RNA-seq files/samples/aliquots, which *existing, audited* source metadata can distinguish candidate biospecimens? Does that information supply a defensible basis for selecting one observation per case?

This is a **metadata evidence audit**, not an actual adjudication. It consumes the already classified RNA cases and Notebook 111's frozen candidate file index; it does not rerun source acquisition, reopen ancestral assignment, reinterpret ambiguous files, or inspect any gene expression or program scores.

The preserved GDC descriptors `sample_type`, `tissue_type` and `tumor_descriptor` may identify disagreements but cannot resolve tumor-region/timepoint differences by themselves. Sample/aliquot submitter IDs are identifiers, not evidence of relative quality. Provider file size, MD5 and generic STAR workflow are **not sequencing QC selection metrics**. Even identical submitted labels do not prove samples are interchangeable.

All 288 case/file identities remain in a **NOT_FROZEN** state. Any future one-per-case decision must use a separately reviewed outcome-blind policy, with enough objective specimen provenance and technical QC. Cases without a defensible selection basis remain unresolved; never select on UUID ordering, file size, gene expression or program score.

In [16]:
# =============================================================================
# Restrict new evidence review to the 288 RNA cases already identified in 800
# =============================================================================

rna_review_cases = rna_cases.loc[
    rna_cases["needs_selection_policy"], [
        "case_uuid", "rna_multiplicity_class", "n_files", "n_samples",
        "n_aliquots", "cross_file_ancestry_conflict",
    ]
].copy()

RNA_SOURCE_EVIDENCE_COLS = [
    "case_uuid", "file_id", "filename", "sample_uuid", "sample_submitter_id",
    "aliquot_uuid", "aliquot_submitter_id", "sample_type", "tissue_type",
    "tumor_descriptor", "primary_site", "disease_type", "project_id",
    "workflow_type", "platform", "size_bytes", "provider_md5",
]
missing = set(RNA_SOURCE_EVIDENCE_COLS) - set(candidate_primary.columns)
if missing:
    raise KeyError("Required existing Notebook 111 metadata absent: " + str(sorted(missing)))

rna_adjudication_files = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq")
    & candidate_primary["case_uuid"].isin(rna_review_cases["case_uuid"]),
    RNA_SOURCE_EVIDENCE_COLS,
].copy()

if (
    rna_review_cases["case_uuid"].duplicated().any()
    or rna_adjudication_files["file_id"].duplicated().any()
    or set(rna_adjudication_files["case_uuid"]) != set(rna_review_cases["case_uuid"])
    or len(rna_adjudication_files) != int(rna_review_cases["n_files"].sum())
):
    raise ValueError("800-B candidate review does not reconcile to 800-A case inventory")

print("Cases under metadata adjudication review:", len(rna_review_cases))
print("Existing primary RNA file records under review:", len(rna_adjudication_files))

Cases under metadata adjudication review: 288
Existing primary RNA file records under review: 649


In [17]:
# =============================================================================
# Quantify per-case specimen descriptors and source-file metadata coverage
# =============================================================================

def nonnull_count(series):
    return int(series.notna().sum())

def unique_nonnull_count(series):
    return int(series.nunique(dropna=True))

evidence = (
    rna_adjudication_files.groupby("case_uuid", dropna=False)
    .agg(
        observed_files=("file_id", "nunique"),
        observed_samples=("sample_uuid", "nunique"),
        observed_aliquots=("aliquot_uuid", "nunique"),
        sample_submitter_ids=("sample_submitter_id", unique_nonnull_count),
        sample_submitter_id_file_coverage=("sample_submitter_id", nonnull_count),
        aliquot_submitter_ids=("aliquot_submitter_id", unique_nonnull_count),
        aliquot_submitter_id_file_coverage=("aliquot_submitter_id", nonnull_count),
        tissue_type_values=("tissue_type", unique_nonnull_count),
        tissue_type_file_coverage=("tissue_type", nonnull_count),
        tumor_descriptor_values=("tumor_descriptor", unique_nonnull_count),
        tumor_descriptor_file_coverage=("tumor_descriptor", nonnull_count),
        source_site_values=("primary_site", unique_nonnull_count),
        source_disease_values=("disease_type", unique_nonnull_count),
        project_values=("project_id", unique_nonnull_count),
        workflow_values=("workflow_type", unique_nonnull_count),
        platform_values=("platform", unique_nonnull_count),
        distinct_provider_md5=("provider_md5", unique_nonnull_count),
    ).reset_index()
)
rna_adjudication_cases = rna_review_cases.merge(
    evidence, on="case_uuid", how="left", validate="one_to_one"
)

if not (
    rna_adjudication_cases["n_files"].eq(rna_adjudication_cases["observed_files"]).all()
    and rna_adjudication_cases["n_samples"].eq(
        rna_adjudication_cases["observed_samples"]
    ).all()
    and rna_adjudication_cases["n_aliquots"].eq(
        rna_adjudication_cases["observed_aliquots"]
    ).all()
):
    raise ValueError("New specimen evidence does not match prior case structure")

display(
    rna_adjudication_cases[[
        "rna_multiplicity_class", "observed_files",
        "tissue_type_file_coverage", "tumor_descriptor_file_coverage",
        "sample_submitter_id_file_coverage",
        "aliquot_submitter_id_file_coverage",
    ]].groupby("rna_multiplicity_class").sum()
)

,observed_files,tissue_type_file_coverage,tumor_descriptor_file_coverage,sample_submitter_id_file_coverage,aliquot_submitter_id_file_coverage
rna_multiplicity_class,,,,,
MULTIPLE_SAMPLES_IN_CASE,619,619,619,619,619
ONE_SAMPLE_MULTIPLE_ALIQUOTS,30,30,30,30,30


In [18]:
# =============================================================================
# Flag metadata discordance and insufficiency, without resolving samples
# =============================================================================

rna_adjudication_cases["source_descriptor_discordance"] = (
    rna_adjudication_cases[[
        "tissue_type_values", "tumor_descriptor_values",
        "source_site_values", "source_disease_values",
        "project_values", "workflow_values", "platform_values",
    ]].gt(1).any(axis=1)
)
rna_adjudication_cases["source_descriptor_incomplete"] = (
    rna_adjudication_cases["tissue_type_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["tumor_descriptor_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["sample_submitter_id_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["aliquot_submitter_id_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
)
rna_adjudication_cases["same_md5_across_distinct_files"] = (
    rna_adjudication_cases["distinct_provider_md5"].lt(
        rna_adjudication_cases["n_files"]
    )
)

rna_adjudication_cases["adjudication_evidence_class"] = np.select(
    [
        rna_adjudication_cases["cross_file_ancestry_conflict"],
        rna_adjudication_cases["source_descriptor_discordance"],
        rna_adjudication_cases["source_descriptor_incomplete"],
        rna_adjudication_cases["rna_multiplicity_class"].eq(
            "MULTIPLE_SAMPLES_IN_CASE"
        ),
    ],
    [
        "CROSS_FILE_CONFLICT_UNRESOLVED",
        "SOURCE_DESCRIPTORS_DISCORDANT_REVIEW",
        "SOURCE_DESCRIPTORS_INCOMPLETE_REVIEW",
        "MULTIPLE_SAMPLES_NO_QC_SELECTION_BASIS",
    ],
    default="MULTIPLE_ALIQUOTS_OR_FILES_NO_QC_SELECTION_BASIS",
)
rna_adjudication_cases["selection_evidence_status"] = (
    "INSUFFICIENT_TO_SELECT_WITHOUT_PROSPECTIVE_QC_POLICY"
)
rna_adjudication_cases["selected_file_id"] = pd.Series(
    pd.NA, index=rna_adjudication_cases.index, dtype="string"
)

display(
    rna_adjudication_cases["adjudication_evidence_class"]
    .value_counts(dropna=False)
    .rename_axis("evidence_class").reset_index(name="n_cases")
)
print("No files selected or excluded; no technical QC read from STAR payloads")

,evidence_class,n_cases
0,MULTIPLE_SAMPLES_NO_QC_SELECTION_BASIS,276
1,MULTIPLE_ALIQUOTS_OR_FILES_NO_QC_SELECTION_BASIS,12


No files selected or excluded; no technical QC read from STAR payloads


In [19]:
# =============================================================================
# Describe the evidence limits by source tumor context without rerunning 111
# =============================================================================

rna_case_context = rna_cases.loc[
    rna_cases["needs_selection_policy"],
    ["case_uuid", "primary_site_values", "disease_type_values"],
].copy()
rna_adjudication_cases = rna_adjudication_cases.merge(
    rna_case_context, on="case_uuid", validate="one_to_one"
)
display(
    rna_adjudication_cases.groupby(
        ["primary_site_values", "rna_multiplicity_class"], dropna=False
    ).agg(
        n_cases=("case_uuid", "nunique"),
        descriptor_discordance=("source_descriptor_discordance", "sum"),
        incomplete_descriptors=("source_descriptor_incomplete", "sum"),
        same_md5_file_groups=("same_md5_across_distinct_files", "sum"),
    )
    .reset_index()
    .sort_values(["n_cases", "primary_site_values"], ascending=[False, True])
    .head(25)
)
print(
    "Cases with source-descriptor disagreement:",
    int(rna_adjudication_cases["source_descriptor_discordance"].sum())
)
print(
    "Cases with at least one missing specimen descriptor field:",
    int(rna_adjudication_cases["source_descriptor_incomplete"].sum())
)
print("True technical replicate identity / relative RNA-seq QC: NOT ASSESSED")

,primary_site_values,rna_multiplicity_class,n_cases,descriptor_discordance,incomplete_descriptors,same_md5_file_groups
13,"Uterus, NOS",MULTIPLE_SAMPLES_IN_CASE,179,0,0,0
4,Kidney,MULTIPLE_SAMPLES_IN_CASE,40,0,0,0
5,Larynx,MULTIPLE_SAMPLES_IN_CASE,19,0,0,0
11,Other and unspecified parts of tongue,MULTIPLE_SAMPLES_IN_CASE,12,0,0,0
12,Pancreas,ONE_SAMPLE_MULTIPLE_ALIQUOTS,12,0,0,0
2,Floor of mouth,MULTIPLE_SAMPLES_IN_CASE,11,0,0,0
3,Gum,MULTIPLE_SAMPLES_IN_CASE,3,0,0,0
7,Oropharynx,MULTIPLE_SAMPLES_IN_CASE,3,0,0,0
9,"Other and ill-defined sites in lip, oral cavit...",MULTIPLE_SAMPLES_IN_CASE,3,0,0,0
6,Lip,MULTIPLE_SAMPLES_IN_CASE,2,0,0,0


Cases with source-descriptor disagreement: 0
Cases with at least one missing specimen descriptor field: 0
True technical replicate identity / relative RNA-seq QC: NOT ASSESSED


In [20]:
# =============================================================================
# Prepare auditable per-file metadata evidence; preserve case status UNRESOLVED
# =============================================================================

rna_adjudication_files = rna_adjudication_files.merge(
    rna_adjudication_cases[
        ["case_uuid", "rna_multiplicity_class", "adjudication_evidence_class",
         "selection_evidence_status"]
    ],
    on="case_uuid", validate="many_to_one"
)
rna_adjudication_files["candidate_selected"] = False
rna_adjudication_files["selection_performed"] = False

rna_adjudication_cases = rna_adjudication_cases.sort_values(
    "case_uuid", kind="stable"
).reset_index(drop=True)
rna_adjudication_files = rna_adjudication_files.sort_values(
    ["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"], kind="stable"
).reset_index(drop=True)

if (
    len(rna_adjudication_cases) != int(rna_cases["needs_selection_policy"].sum())
    or rna_adjudication_cases["selected_file_id"].notna().any()
    or rna_adjudication_files["candidate_selected"].any()
):
    raise ValueError("Evidence-only 800-B unexpectedly selected or lost cases")

print("Case-level adjudication evidence rows:", len(rna_adjudication_cases))
print("File-level adjudication evidence rows:", len(rna_adjudication_files))
print("Selection eligibility rule: NOT_FROZEN")

Case-level adjudication evidence rows: 288
File-level adjudication evidence rows: 649
Selection eligibility rule: NOT_FROZEN


## PREFLIGHT-C — Source specimen provenance and technical QC *availability* (not adjudication)

**Question.** Does the **preserved acquisition-time GDC RNA metadata snapshot** contain sufficiently specific collection, spatial, preparation or comparable QC information to distinguish the already identified 649 RNA files from the 288 cases requiring review?

This stage first fingerprints the exact metadata snapshot already reported by Notebook 111 and examines **only those 649 audited file IDs**. For each file, additional fields are read **only from the case → sample → portion → analyte → aliquot path matching the exact three UUIDs already resolved in Notebook 111**. The step does not redo ancestry resolution or rescue the ambiguous records.

The audit distinguishes **key present**, **informative scalar value** (excluding common unknown/not-reported placeholders), and **multiple different values on matching hierarchical paths**. A GDC file creation date is **not** a specimen collection date. A sample submitter ID, portion number or concentration is **not** a QC score; generic `sample_type`, `tissue_type` or `tumor_descriptor` does **not** establish a tumor sampling region. An apparently uniform field does not prove identical tumor regions/timepoints.

**QC scope:** structural *availability of metadata fields* only. No processed STAR expression payloads, source QC reports, alignment logs or newly queried GDC information are inspected. Absent keys mean **not present in this audited snapshot**, *not* absent from the provider. No sample is selected or excluded; RNA file quality and relative quality remain unresolved.

In [21]:
# =============================================================================
# Read only the RNA acquisition-time metadata snapshot already frozen by 111
# =============================================================================

RNA_GDC_METADATA_PATH = (
    Paths.config / "manifests" / "cptac3" / "gdc_metadata_cptac3_rnaseq.json"
)
expected_rna_metadata_sha = audit_summary[
    "source_fingerprints"
]["rnaseq"]["metadata"]["sha256"]
if calculate_sha256(RNA_GDC_METADATA_PATH) != expected_rna_metadata_sha:
    raise ValueError("RNA acquisition-time GDC metadata differs from Notebook 111")

with RNA_GDC_METADATA_PATH.open("r", encoding="utf-8") as handle:
    rna_snapshot = json.load(handle)
if not isinstance(rna_snapshot, dict) or not isinstance(
    rna_snapshot.get("data"), dict
):
    raise ValueError("Unexpected RNA metadata snapshot envelope")
rna_metadata_hits = rna_snapshot["data"].get("hits")
if not isinstance(rna_metadata_hits, list):
    raise ValueError("RNA metadata snapshot has no data.hits list")

# The snapshot is already source-audited by 111: index only 800-B's 649 files.
review_file_ids = set(rna_adjudication_files["file_id"])
rna_c_hits = {}
for hit in rna_metadata_hits:
    file_uuid = hit.get("file_id") or hit.get("id")
    if file_uuid in review_file_ids:
        if file_uuid in rna_c_hits:
            raise ValueError("Duplicate file ID in reviewed RNA snapshot subset")
        rna_c_hits[file_uuid] = hit
if set(rna_c_hits) != review_file_ids:
    raise ValueError("Missing 800-B file IDs in frozen RNA metadata snapshot")

print("Exact reviewed GDC metadata files:", len(rna_c_hits))
print("RNA metadata snapshot SHA-256: PASS (Notebook 111 reference)")
print("No API query, source re-acquisition or STAR payload QC scan")

Exact reviewed GDC metadata files: 649
RNA metadata snapshot SHA-256: PASS (Notebook 111 reference)
No API query, source re-acquisition or STAR payload QC scan


In [22]:
# =============================================================================
# Locate exact already-audited triplets to expose additional source fields
# =============================================================================

def exact_source_nodes(hit, case_uuid, sample_uuid, aliquot_uuid):
    paths = []
    for case in hit.get("cases") or []:
        if case.get("case_id") != case_uuid:
            continue
        for sample in case.get("samples") or []:
            if sample.get("sample_id") != sample_uuid:
                continue
            for portion in sample.get("portions") or []:
                for analyte in portion.get("analytes") or []:
                    for aliquot in analyte.get("aliquots") or []:
                        if aliquot.get("aliquot_id") == aliquot_uuid:
                            paths.append({
                                "sample": sample,
                                "portion": portion,
                                "analyte": analyte,
                                "aliquot": aliquot,
                            })
    return paths

rna_c_node_maps = {}
rna_c_path_rows = []
for row in rna_adjudication_files.itertuples(index=False):
    hit = rna_c_hits[row.file_id]
    paths = exact_source_nodes(
        hit, row.case_uuid, row.sample_uuid, row.aliquot_uuid
    )
    if not paths:
        raise ValueError(
            "Notebook 111 audited case/sample/aliquot path not found for "
            + str(row.file_id)
        )
    rna_c_node_maps[row.file_id] = {
        "file": [hit],
        "analysis": [hit.get("analysis")] if isinstance(
            hit.get("analysis"), dict
        ) else [],
        **{
            scope: [entry[scope] for entry in paths]
            for scope in ("sample", "portion", "analyte", "aliquot")
        },
    }
    rna_c_path_rows.append({
        "file_id": row.file_id,
        "case_uuid": row.case_uuid,
        "sample_uuid": row.sample_uuid,
        "aliquot_uuid": row.aliquot_uuid,
        "exact_hierarchy_path_rows": len(paths),
    })

rna_c_exact_path_index = pd.DataFrame(rna_c_path_rows)
print("Exact previously audited triplets represented:",
      len(rna_c_exact_path_index))
print("Files with >1 exact hierarchy row (not distinct parents):",
      int(rna_c_exact_path_index["exact_hierarchy_path_rows"].gt(1).sum()))

Exact previously audited triplets represented: 649
Files with >1 exact hierarchy row (not distinct parents): 0


In [23]:
# =============================================================================
# Define conservative, testable source metadata availability domains
# =============================================================================

RNA_C_FIELDS = {
    "generic_sample_descriptors": [
        ("sample", "sample_type"), ("sample", "tissue_type"),
        ("sample", "tumor_descriptor"),
    ],
    "collection_timing": [
        ("sample", "days_to_collection"),
        ("sample", "days_to_sample_procurement"),
        ("sample", "time_between_clamping_and_freezing"),
        ("sample", "time_between_excision_and_freezing"),
    ],
    "sampling_location_detail": [
        ("sample", "tumor_region"), ("sample", "tumor_subsite"),
        ("sample", "sample_collection_site"), ("sample", "anatomic_site"),
        ("sample", "specimen_laterality"),
    ],
    "processing_provenance": [
        ("sample", "method_of_sample_procurement"),
        ("sample", "preservation_method"), ("sample", "is_ffpe"),
        ("portion", "is_ffpe"), ("portion", "portion_number"),
        ("portion", "weight"), ("analyte", "analyte_type"),
        ("analyte", "concentration"), ("aliquot", "concentration"),
    ],
    "identifiers_not_qc": [
        ("sample", "submitter_id"), ("portion", "submitter_id"),
        ("analyte", "submitter_id"), ("aliquot", "submitter_id"),
    ],
    "file_processing_dates_not_collection": [
        ("file", "created_datetime"), ("file", "updated_datetime"),
        ("analysis", "analysis_id"), ("analysis", "workflow_type"),
        ("analysis", "workflow_version"),
    ],
    "quantitative_qc_metadata_probes": [
        ("file", "total_reads"), ("file", "mapped_reads"),
        ("file", "average_read_length"), ("file", "average_base_quality"),
        ("analysis", "total_reads"), ("analysis", "mapped_reads"),
        ("analysis", "mapping_rate"), ("analysis", "duplication_rate"),
        ("analysis", "percent_aligned_reads"),
    ],
}

# Presence here means present in the preserved projection, not in all GDC data.
PLACEHOLDER_STRINGS = {
    "", "unknown", "not reported", "not available", "not applicable",
    "n/a", "na", "null", "none", "unspecified", "--",
}

def informative_scalar(value):
    if value is None or isinstance(value, (list, tuple, dict)):
        return False
    if isinstance(value, str):
        return value.strip().casefold() not in PLACEHOLDER_STRINGS
    if isinstance(value, (bool, int, float)):
        return bool(pd.notna(value))
    return False

def source_value_status(nodes, key):
    values = [
        item[key] for item in nodes
        if isinstance(item, dict) and key in item
    ]
    useful = {str(v).strip() for v in values if informative_scalar(v)}
    return bool(values), bool(useful), len(useful) > 1

all_probes = {
    (scope, key)
    for group in RNA_C_FIELDS.values()
    for scope, key in group
}
print("Explicit availability probes:", len(all_probes))
print("No thresholds for donor identity, specimen equivalence or quality")

Explicit availability probes: 39
No thresholds for donor identity, specimen equivalence or quality


In [24]:
# =============================================================================
# Inventory observed scalar snapshot keys and quantitative-QC probe coverage
# =============================================================================

observed_scalar_keys = set()
for node_map in rna_c_node_maps.values():
    for scope, nodes in node_map.items():
        for node in nodes:
            if isinstance(node, dict):
                observed_scalar_keys.update(
                    (scope, key) for key, value in node.items()
                    if not isinstance(value, (dict, list, tuple))
                )

# Inventory includes actual fields, plus explicit absent probes for auditability.
all_field_keys = sorted(all_probes | observed_scalar_keys)
rna_c_field_records = []
for scope, key in all_field_keys:
    n_key, n_informative, n_discordant = 0, 0, 0
    for nodes in rna_c_node_maps.values():
        present, informative, discordant = source_value_status(
            nodes[scope], key
        )
        n_key += int(present)
        n_informative += int(informative)
        n_discordant += int(discordant)
    assigned_domains = sorted(
        name for name, fields in RNA_C_FIELDS.items()
        if (scope, key) in fields
    )
    rna_c_field_records.append({
        "scope": scope, "field": key,
        "planned_domains": "|".join(assigned_domains),
        "observed_scalar_key": (scope, key) in observed_scalar_keys,
        "n_files_with_key": n_key,
        "n_files_with_informative_scalar": n_informative,
        "n_files_with_multiple_informative_values": n_discordant,
        "n_reviewed_files": len(rna_c_node_maps),
    })
rna_c_field_coverage = pd.DataFrame(rna_c_field_records)
display(
    rna_c_field_coverage.loc[
        rna_c_field_coverage["planned_domains"].ne("")
    ].sort_values(["planned_domains", "scope", "field"])
)
print(
    "Observed scalar fields not in explicit probe list:",
    len(observed_scalar_keys - all_probes)
)

,scope,field,planned_domains,observed_scalar_key,n_files_with_key,n_files_with_informative_scalar,n_files_with_multiple_informative_values,n_reviewed_files
34,sample,days_to_collection,collection_timing,False,0,0,0,649
35,sample,days_to_sample_procurement,collection_timing,False,0,0,0,649
44,sample,time_between_clamping_and_freezing,collection_timing,False,0,0,0,649
45,sample,time_between_excision_and_freezing,collection_timing,False,0,0,0,649
3,analysis,analysis_id,file_processing_dates_not_collection,False,0,0,0,649
9,analysis,workflow_type,file_processing_dates_not_collection,True,649,649,0,649
10,analysis,workflow_version,file_processing_dates_not_collection,True,649,649,0,649
17,file,created_datetime,file_processing_dates_not_collection,False,0,0,0,649
28,file,updated_datetime,file_processing_dates_not_collection,False,0,0,0,649
41,sample,sample_type,generic_sample_descriptors,True,649,649,0,649


Observed scalar fields not in explicit probe list: 11


In [25]:
# =============================================================================
# Flag usable specimen/QC metadata *availability* for each of 649 files
# =============================================================================

rna_c_file_rows = []
for row in rna_adjudication_files.itertuples(index=False):
    nodes = rna_c_node_maps[row.file_id]
    new_record = {
        "file_id": row.file_id,
        "case_uuid": row.case_uuid,
        "sample_uuid": row.sample_uuid,
        "aliquot_uuid": row.aliquot_uuid,
        "rna_multiplicity_class": row.rna_multiplicity_class,
    }
    for domain, probes in RNA_C_FIELDS.items():
        statuses = [
            source_value_status(nodes[scope], key)
            for scope, key in probes
        ]
        new_record["informative_" + domain] = any(
            informative for _, informative, _ in statuses
        )
        new_record["discordant_" + domain] = any(
            discordant for _, _, discordant in statuses
        )
        new_record["n_informative_fields_" + domain] = sum(
            informative for _, informative, _ in statuses
        )
    rna_c_file_rows.append(new_record)

rna_c_file_availability = pd.DataFrame(rna_c_file_rows).merge(
    rna_c_exact_path_index[["file_id", "exact_hierarchy_path_rows"]],
    on="file_id", validate="one_to_one"
).sort_values(["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"])
rna_c_file_availability = rna_c_file_availability.reset_index(drop=True)

summary_fields = [
    "informative_" + name for name in RNA_C_FIELDS
]
display(
    rna_c_file_availability.groupby("rna_multiplicity_class")[
        summary_fields
    ].sum()
)
print("Non-informative placeholders are not counted as usable values")
print("Provider-level and processed STAR QC: NOT TESTED")

,informative_generic_sample_descriptors,informative_collection_timing,informative_sampling_location_detail,informative_processing_provenance,informative_identifiers_not_qc,informative_file_processing_dates_not_collection,informative_quantitative_qc_metadata_probes
rna_multiplicity_class,,,,,,,
MULTIPLE_SAMPLES_IN_CASE,619,0,0,0,619,619,0
ONE_SAMPLE_MULTIPLE_ALIQUOTS,30,0,0,0,30,30,0


Non-informative placeholders are not counted as usable values
Provider-level and processed STAR QC: NOT TESTED


In [26]:
# =============================================================================
# Summarize 288-case coverage without adjudicating within-case candidates
# =============================================================================

rna_c_case_coverage = (
    rna_c_file_availability.groupby("case_uuid")
    .agg(
        n_files=("file_id", "nunique"),
        n_exact_hierarchy_rows=("exact_hierarchy_path_rows", "sum"),
        **{
            "n_files_informative_" + name: (
                "informative_" + name, "sum"
            )
            for name in RNA_C_FIELDS
        },
        **{
            "n_files_discordant_" + name: (
                "discordant_" + name, "sum"
            )
            for name in RNA_C_FIELDS
        },
    )
    .reset_index()
    .merge(
        rna_review_cases[[
            "case_uuid", "rna_multiplicity_class", "n_files"
        ]].rename(columns={"n_files": "audited_n_files"}),
        on="case_uuid", validate="one_to_one"
    )
)
if (
    len(rna_c_case_coverage) != len(rna_review_cases)
    or not rna_c_case_coverage["n_files"].eq(
        rna_c_case_coverage["audited_n_files"]
    ).all()
):
    raise ValueError("800-C changed the source-snapshot review denominator")
rna_c_case_coverage["selection_policy_status"] = "NOT_FROZEN"
rna_c_case_coverage = rna_c_case_coverage.sort_values(
    "case_uuid", kind="stable"
).reset_index(drop=True)

display(
    rna_c_case_coverage.groupby("rna_multiplicity_class")
    .agg(
        cases=("case_uuid", "nunique"),
        with_any_collection_timing=(
            "n_files_informative_collection_timing", lambda v: int(v.gt(0).sum())
        ),
        with_all_collection_timing=(
            "n_files_informative_collection_timing",
            lambda v: int(v.eq(
                rna_c_case_coverage.loc[v.index, "n_files"]
            ).sum()),
        ),
        with_any_location_detail=(
            "n_files_informative_sampling_location_detail",
            lambda v: int(v.gt(0).sum())
        ),
        with_any_quantitative_qc_metadata=(
            "n_files_informative_quantitative_qc_metadata_probes",
            lambda v: int(v.gt(0).sum())
        ),
    )
)
print("All 288 cases remain unresolved for one-record-per-case selection")

,cases,with_any_collection_timing,with_all_collection_timing,with_any_location_detail,with_any_quantitative_qc_metadata
rna_multiplicity_class,,,,,
MULTIPLE_SAMPLES_IN_CASE,276,0,0,0,0
ONE_SAMPLE_MULTIPLE_ALIQUOTS,12,0,0,0,0


All 288 cases remain unresolved for one-record-per-case selection


In [27]:
# =============================================================================
# Enforce no-selection and scope guarantees for this availability-only stage
# =============================================================================

if (
    len(rna_c_file_availability) != len(rna_adjudication_files)
    or rna_c_file_availability["file_id"].duplicated().any()
    or rna_c_case_coverage["case_uuid"].duplicated().any()
    or rna_adjudication_files["candidate_selected"].any()
    or rna_adjudication_cases["selected_file_id"].notna().any()
    or rna_c_case_coverage["selection_policy_status"].ne("NOT_FROZEN").any()
):
    raise ValueError("800-C must not select or lose RNA source records")

print("PREFLIGHT-C source-metadata availability inventory: READY TO PERSIST")
print("Observed metadata availability is not evidence of relative sample QC")
print("Biological selection / program scoring: NOT PERFORMED")

PREFLIGHT-C source-metadata availability inventory: READY TO PERSIST
Observed metadata availability is not evidence of relative sample QC
Biological selection / program scoring: NOT PERFORMED


## PREFLIGHT-D — Targeted live-GDC provenance and QC feasibility pilot

PREFLIGHT-C found no informative *probed* collection-time, specimen-localization or quantitative-QC variables in its preserved acquisition-time metadata projection. **This does not establish provider-wide absence.** The GDC data dictionary includes, for example, `samples.biospecimen_anatomic_site`, `samples.biospecimen_laterality`, `samples.portions.analytes.rna_integrity_number`, and nested read-group QC structures. Availability and suitability must be checked on exact identities, rather than inferred from the schema.

**Pilot only, before scaling to 649 files.** Select three cases each from the already observed RNA multiplicity strata `Uterus, NOS`, `Kidney`, `Larynx` (multiple samples) and `Pancreas` (multiple aliquots), by stable UUID ordering **solely for technical diagnostic sampling**. This is not a tumor-selection, quality or statistical analysis rule. No additional source file UUID is permitted. Query only those pilot file UUIDs from the public GDC `/files` endpoint and preserve a *separate*, immutable-on-rerun query snapshot. Failed API retrieval must never be interpreted as missing provider fields.

**Identity and linkage first.** Reconcile live response file UUID, filename, size and provider MD5 against Notebook 111's already audited file index, without rereading original payloads. Match any newly returned sample/portion/analyte/aliquot data to the **same exact case/sample/aliquot triplets resolved by Notebook 111**; case-level membership or presence of extra sample UUIDs is not new file-to-sample evidence. Flag discrepancies and missing nested paths explicitly. Read-group metadata can be associated with the returned file's analysis but may be absent, unlinked to the input library, or incomparable; availability does not demonstrate quality equivalence.

**Decision gate.** Count candidate fields with exact-link informative values, discrepancies, and within-case differentiation; do not select a file, impose QC thresholds, recompute sample ancestry, score any program, or automatically expand the live query to all 649 files. The next scale/no-scale decision is manual after examining this diagnostic pilot. Source release/timepoint drift is recorded as a risk; it is not silently combined with the original GDC snapshot.

In [28]:
# =============================================================================
# Freeze a bounded, descriptive GDC pilot from the existing 288 case review
# =============================================================================

import hashlib
import time
from datetime import datetime, timezone

import requests

RNA_D_PILOT_STRATA = (
    ("MULTIPLE_SAMPLES_IN_CASE", "Uterus, NOS"),
    ("MULTIPLE_SAMPLES_IN_CASE", "Kidney"),
    ("MULTIPLE_SAMPLES_IN_CASE", "Larynx"),
    ("ONE_SAMPLE_MULTIPLE_ALIQUOTS", "Pancreas"),
)
rna_d_pilot_cases = pd.concat([
    rna_adjudication_cases.loc[
        rna_adjudication_cases["rna_multiplicity_class"].eq(kind)
        & rna_adjudication_cases["primary_site_values"].eq(site)
    ].sort_values("case_uuid", kind="stable").head(3)
    for kind, site in RNA_D_PILOT_STRATA
], ignore_index=True)

if (
    len(rna_d_pilot_cases) != 12
    or rna_d_pilot_cases["case_uuid"].duplicated().any()
):
    raise ValueError("Unexpected 12-case descriptive sampling strata")

rna_d_pilot_files = rna_adjudication_files.loc[
    rna_adjudication_files["case_uuid"].isin(
        rna_d_pilot_cases["case_uuid"]
    )
].copy()
rna_d_file_ids = sorted(rna_d_pilot_files["file_id"].tolist())
if (
    len(rna_d_file_ids) != len(set(rna_d_file_ids))
    or not set(rna_d_file_ids).issubset(
        set(rna_adjudication_files["file_id"])
    )
):
    raise ValueError("PREFLIGHT-D pilot expanded beyond Notebook 800-B file identities")

print("Pilot cases:", len(rna_d_pilot_cases))
print("Pilot exact files:", len(rna_d_file_ids))
display(rna_d_pilot_cases.groupby(
    ["rna_multiplicity_class", "primary_site_values"]
).size().rename("n_cases").reset_index())
print("Stable UUID ordering is diagnostic sampling only, NOT file eligibility")

Pilot cases: 12
Pilot exact files: 29


,rna_multiplicity_class,primary_site_values,n_cases
0,MULTIPLE_SAMPLES_IN_CASE,Kidney,3
1,MULTIPLE_SAMPLES_IN_CASE,Larynx,3
2,MULTIPLE_SAMPLES_IN_CASE,"Uterus, NOS",3
3,ONE_SAMPLE_MULTIPLE_ALIQUOTS,Pancreas,3


Stable UUID ordering is diagnostic sampling only, NOT file eligibility


In [29]:
# =============================================================================
# Diagnose preserved v1 structure; capture separate exact-file AND exact-case v2
# =============================================================================

RNA_D_PRIOR_SNAPSHOT_PATH = OUTPUT_PATHS["rna_d_live_gdc_snapshot"]
RNA_D_SNAPSHOT_PATH = OUTPUT_PATHS["rna_d_live_gdc_snapshot_v2"]
RNA_D_GDC_URL = "https://api.gdc.cancer.gov/files"
RNA_D_CASES_URL = "https://api.gdc.cancer.gov/cases"

if RNA_D_PRIOR_SNAPSHOT_PATH.is_file():
    with RNA_D_PRIOR_SNAPSHOT_PATH.open("r", encoding="utf-8") as handle:
        rna_d_previous = json.load(handle)
    previous_hits = rna_d_previous.get("hits") or []
    if len(previous_hits) != len(rna_d_file_ids):
        raise ValueError("Previous D pilot snapshot has an unexpected file count")
    print("Preserved v1 snapshot:", project_relative_path(RNA_D_PRIOR_SNAPSHOT_PATH))
    print("Previous file records with case_id in nested cases:", sum(
        any(isinstance(case, dict) and case.get("case_id")
            for case in hit.get("cases") or [])
        for hit in previous_hits
    ))
    print("Previous file records containing any nested sample_id:", sum(
        any(isinstance(sample, dict) and sample.get("sample_id")
            for case in hit.get("cases") or []
            for sample in case.get("samples") or [])
        for hit in previous_hits
    ))
else:
    print("No previous v1 snapshot available; do not infer its structure")

# Files endpoint: request the exact hierarchy identifiers explicitly.
RNA_D_API_FIELDS = [
    "file_id", "file_name", "file_size", "md5sum", "access",
    "data_type", "data_category", "cases.case_id",
    "cases.samples.sample_id", "cases.samples.portions.portion_id",
    "cases.samples.portions.analytes.analyte_id",
    "cases.samples.portions.analytes.aliquots.aliquot_id",
    "proportion_reads_mapped", "proportion_reads_duplicated",
    "total_reads", "average_base_quality",
]
RNA_D_API_EXPAND = [
    "associated_entities",
    "cases.samples", "cases.samples.portions",
    "cases.samples.portions.analytes",
    "cases.samples.portions.analytes.aliquots",
    "analysis", "analysis.metadata.read_groups",
    "analysis.metadata.read_groups.read_group_qcs",
]
rna_d_file_query = {
    "filters": {
        "op": "in",
        "content": {"field": "file_id", "value": rna_d_file_ids},
    },
    "fields": ",".join(RNA_D_API_FIELDS),
    "expand": ",".join(RNA_D_API_EXPAND),
    "format": "JSON",
    "size": len(rna_d_file_ids),
}

# Cases endpoint: an independent projection of the EXACT 12 case UUIDs.
# Case-scoped data NEVER infer file ancestry without a direct file-aliquot link.
rna_d_case_ids = sorted(rna_d_pilot_cases["case_uuid"].tolist())
RNA_D_CASE_FIELDS = [
    "case_id", "samples.sample_id",
    "samples.biospecimen_anatomic_site", "samples.biospecimen_laterality",
    "samples.days_to_collection", "samples.method_of_sample_procurement",
    "samples.preservation_method",
    "samples.portions.portion_id",
    "samples.portions.analytes.analyte_id",
    "samples.portions.analytes.rna_integrity_number",
    "samples.portions.analytes.aliquots.aliquot_id",
]
RNA_D_CASE_EXPAND = [
    "samples", "samples.portions", "samples.portions.analytes",
    "samples.portions.analytes.aliquots",
]
rna_d_case_query = {
    "filters": {
        "op": "in",
        "content": {"field": "case_id", "value": rna_d_case_ids},
    },
    "fields": ",".join(RNA_D_CASE_FIELDS),
    "expand": ",".join(RNA_D_CASE_EXPAND),
    "format": "JSON",
    "size": len(rna_d_case_ids),
}
rna_d_query_contract = {
    "files": rna_d_file_query,
    "cases": rna_d_case_query,
}
rna_d_query_sha = hashlib.sha256(
    json.dumps(
        rna_d_query_contract, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
).hexdigest()

def gdc_exact_hits(session, endpoint, payload, expected_ids, id_field):
    for attempt in range(3):
        try:
            response = session.post(endpoint, json=payload, timeout=(20, 120))
            response.raise_for_status()
            result = response.json()
            hits = (result.get("data") or {}).get("hits")
            if not isinstance(hits, list):
                raise ValueError("GDC endpoint returned no data.hits list")
            observed = [item.get(id_field) or item.get("id") for item in hits]
            if (
                len(observed) != len(expected_ids)
                or sorted(observed) != expected_ids
                or len(set(observed)) != len(expected_ids)
            ):
                raise ValueError("GDC exact-query identity/count discrepancy")
            return sorted(hits, key=lambda item: item.get(id_field) or item.get("id"))
        except (requests.RequestException, ValueError, TypeError) as error:
            if attempt == 2:
                raise RuntimeError(
                    "GDC targeted retrieval failed; not evidence of field absence"
                ) from error
            time.sleep(2 ** attempt)
    raise RuntimeError("Unreachable retrieval state")

if RNA_D_SNAPSHOT_PATH.is_file():
    with RNA_D_SNAPSHOT_PATH.open("r", encoding="utf-8") as handle:
        rna_d_snapshot = json.load(handle)
    if (
        rna_d_snapshot.get("query_sha256") != rna_d_query_sha
        or rna_d_snapshot.get("originating_111_rnaseq_metadata_sha256")
        != expected_rna_metadata_sha
    ):
        raise ValueError("Cached v2 GDC snapshot does not match its exact queries")
    print("Reusing immutable-on-rerun PREFLIGHT-D v2 snapshot")
else:
    with requests.Session() as session:
        file_hits = gdc_exact_hits(
            session, RNA_D_GDC_URL, rna_d_file_query, rna_d_file_ids, "file_id"
        )
        case_hits = gdc_exact_hits(
            session, RNA_D_CASES_URL, rna_d_case_query, rna_d_case_ids, "case_id"
        )
    rna_d_snapshot = {
        "version": 2,
        "source": "NCI GDC exact-file and exact-case API projections",
        "captured_at_utc": datetime.now(timezone.utc).isoformat(),
        "queries": rna_d_query_contract,
        "query_sha256": rna_d_query_sha,
        "originating_111_rnaseq_metadata_sha256": expected_rna_metadata_sha,
        "v1_snapshot_sha256": (
            calculate_sha256(RNA_D_PRIOR_SNAPSHOT_PATH)
            if RNA_D_PRIOR_SNAPSHOT_PATH.is_file() else None
        ),
        "hits": file_hits,
        "case_hits": case_hits,
    }
    RNA_D_SNAPSHOT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with RNA_D_SNAPSHOT_PATH.open("x", encoding="utf-8") as handle:
        json.dump(rna_d_snapshot, handle, indent=2, ensure_ascii=False)
        handle.write("\n")
    print("Saved separate PREFLIGHT-D v2 snapshot:",
          project_relative_path(RNA_D_SNAPSHOT_PATH))

rna_d_hits = rna_d_snapshot.get("hits")
rna_d_case_hits = rna_d_snapshot.get("case_hits")
if not isinstance(rna_d_hits, list) or not isinstance(rna_d_case_hits, list):
    raise ValueError("PREFLIGHT-D v2 snapshot lacks one of its two source arrays")
rna_d_hit_by_id = {
    hit.get("file_id") or hit.get("id"): hit for hit in rna_d_hits
}
rna_d_case_by_id = {
    hit.get("case_id") or hit.get("id"): hit for hit in rna_d_case_hits
}
if (
    len(rna_d_hit_by_id) != len(rna_d_file_ids)
    or set(rna_d_hit_by_id) != set(rna_d_file_ids)
    or len(rna_d_case_by_id) != len(rna_d_case_ids)
    or set(rna_d_case_by_id) != set(rna_d_case_ids)
):
    raise ValueError("Cached v2 source file/case UUID sets do not reconcile")
print("V2 file UUIDs:", len(rna_d_hit_by_id),
      "| V2 case UUIDs:", len(rna_d_case_by_id))
print("V2 snapshot SHA-256:", calculate_sha256(RNA_D_SNAPSHOT_PATH))

Preserved v1 snapshot: data/interim/metadata/800_preflight_d_gdc_targeted_pilot.json
Previous file records with case_id in nested cases: 0
Previous file records containing any nested sample_id: 29
Reusing immutable-on-rerun PREFLIGHT-D v2 snapshot
V2 file UUIDs: 29 | V2 case UUIDs: 12
V2 snapshot SHA-256: 297c48c0b6be7979b56e12fb67853cbc32ee75017d79eafb28b4cba307d25c31


In [30]:
# =============================================================================
# Reconcile file-aliquot association before accepting exact case sample metadata
# =============================================================================

rna_d_nodes = {}
rna_d_identity_rows = []
for row in rna_d_pilot_files.itertuples(index=False):
    hit = rna_d_hit_by_id[row.file_id]
    case_hit = rna_d_case_by_id[row.case_uuid]
    identity_matches = (
        hit.get("file_name") == row.filename
        and str(hit.get("file_size")) == str(row.size_bytes)
        and str(hit.get("md5sum", "")).lower() == str(row.provider_md5).lower()
    )
    if not identity_matches:
        raise ValueError(
            "Live GDC file identity differs from Notebook 111: " + str(row.file_id)
        )

    direct = hit.get("associated_entities")
    if not isinstance(direct, list):
        direct_status = "UNAVAILABLE_IN_LIVE_RESPONSE"
    else:
        direct_links = [
            entity for entity in direct
            if entity.get("entity_type") == "aliquot"
        ]
        exact = [
            entity for entity in direct_links
            if entity.get("entity_id") == row.aliquot_uuid
            and entity.get("case_id") == row.case_uuid
        ]
        direct_status = (
            "EXACT_MATCH" if len(direct_links) == len(exact) == 1
            else "DIRECT_ASSOCIATION_DISAGREEMENT"
        )

    file_paths = exact_source_nodes(
        hit, row.case_uuid, row.sample_uuid, row.aliquot_uuid
    )
    # /cases contains every sample for its case; only the precise frozen
    # triplet (case, sample, aliquot) can be considered AFTER direct matching.
    case_paths = exact_source_nodes(
        {"cases": [case_hit]},
        row.case_uuid, row.sample_uuid, row.aliquot_uuid
    )

    if direct_status != "EXACT_MATCH":
        chosen_paths, path_source = [], "BLOCKED_DIRECT_FILE_ALIQUOT_LINK"
    elif case_paths:
        chosen_paths, path_source = case_paths, "CASES_ENDPOINT_EXACT_TRIPLET"
    elif file_paths:
        chosen_paths, path_source = file_paths, "FILES_ENDPOINT_EXACT_TRIPLET"
    else:
        chosen_paths, path_source = [], "BLOCKED_NO_EXACT_TRIPLET"

    if chosen_paths:
        node_map = {
            scope: [item[scope] for item in chosen_paths]
            for scope in ("sample", "portion", "analyte", "aliquot")
        }
    else:
        node_map = {scope: [] for scope in (
            "sample", "portion", "analyte", "aliquot"
        )}

    analysis = hit.get("analysis")
    metadata = analysis.get("metadata") if isinstance(analysis, dict) else None
    read_groups = (
        metadata.get("read_groups") or []
        if isinstance(metadata, dict) else []
    )
    if not isinstance(read_groups, list):
        read_groups = []
    qcs = [
        qc for group in read_groups if isinstance(group, dict)
        for qc in (group.get("read_group_qcs") or [])
        if isinstance(qc, dict)
    ]
    node_map.update({
        "file": [hit],
        "analysis": [analysis] if isinstance(analysis, dict) else [],
        "read_group_qc": qcs,
    })
    rna_d_nodes[row.file_id] = node_map
    rna_d_identity_rows.append({
        "file_id": row.file_id,
        "case_uuid": row.case_uuid,
        "sample_uuid": row.sample_uuid,
        "aliquot_uuid": row.aliquot_uuid,
        "rna_multiplicity_class": row.rna_multiplicity_class,
        "primary_site": row.primary_site,
        "live_file_identity": "EXACT_MATCH",
        "live_direct_association": direct_status,
        "n_exact_paths_in_files_response": len(file_paths),
        "n_exact_paths_in_cases_response": len(case_paths),
        "n_exact_hierarchy_paths": len(chosen_paths),
        "exact_biospecimen_path_source": path_source,
        "n_read_groups": len(read_groups),
        "n_read_group_qcs": len(qcs),
    })

rna_d_identity = pd.DataFrame(rna_d_identity_rows).sort_values(
    ["case_uuid", "file_id"]
).reset_index(drop=True)
rna_d_exact_link_n = int(
    rna_d_identity["n_exact_hierarchy_paths"].gt(0).sum()
)
rna_d_exact_link_status = (
    "COMPLETE_EXACT_TRIPLET_RECOVERED"
    if rna_d_exact_link_n == len(rna_d_file_ids)
    else "BLOCKED_INCOMPLETE_EXACT_TRIPLET"
)
display(rna_d_identity.groupby(
    ["primary_site", "exact_biospecimen_path_source"]
).agg(
    n_files=("file_id", "nunique"),
    n_files_with_file_hierarchy=(
        "n_exact_paths_in_files_response", lambda s: int(s.gt(0).sum())
    ),
    n_files_with_case_hierarchy=(
        "n_exact_paths_in_cases_response", lambda s: int(s.gt(0).sum())
    ),
    n_files_with_qc_records=(
        "n_read_group_qcs", lambda s: int(s.gt(0).sum())
    ),
).reset_index())
print("Exact-path links:", rna_d_exact_link_n, "/", len(rna_d_file_ids))
print("PREFLIGHT-D provenance-link status:", rna_d_exact_link_status)
if rna_d_exact_link_status != "COMPLETE_EXACT_TRIPLET_RECOVERED":
    print(
        "BLOCKER: unavailable exact-linked biospecimen fields are NOT "
        "interpretable as provider-wide absence or adequate QC."
    )

,primary_site,exact_biospecimen_path_source,n_files,n_files_with_file_hierarchy,n_files_with_case_hierarchy,n_files_with_qc_records
0,Kidney,CASES_ENDPOINT_EXACT_TRIPLET,9,9,9,0
1,Larynx,CASES_ENDPOINT_EXACT_TRIPLET,6,6,6,0
2,Pancreas,CASES_ENDPOINT_EXACT_TRIPLET,8,8,8,0
3,"Uterus, NOS",CASES_ENDPOINT_EXACT_TRIPLET,6,6,6,0


Exact-path links: 29 / 29
PREFLIGHT-D provenance-link status: COMPLETE_EXACT_TRIPLET_RECOVERED


In [31]:
# =============================================================================
# Audit new provider fields and nested QC availability without quality ranking
# =============================================================================

RNA_D_PROBES = {
    "spatial_specimen": [
        ("sample", "biospecimen_anatomic_site"),
        ("sample", "biospecimen_laterality"),
        ("sample", "tissue_collection_type"),
        ("sample", "specimen_type"),
    ],
    "collection_provenance": [
        ("sample", "days_to_collection"),
        ("sample", "days_to_sample_procurement"),
        ("sample", "method_of_sample_procurement"),
        ("sample", "preservation_method"),
        ("sample", "freezing_method"),
        ("sample", "time_between_excision_and_freezing"),
    ],
    "molecular_preparation": [
        ("portion", "is_ffpe"), ("portion", "portion_number"),
        ("analyte", "analyte_type"),
        ("analyte", "rna_integrity_number"),
        ("analyte", "ribosomal_rna_28s_18s_ratio"),
        ("analyte", "a260_a280_ratio"),
        ("analyte", "concentration"),
        ("aliquot", "source_center"),
    ],
    "file_level_qc_probes": [
        ("file", "proportion_reads_mapped"),
        ("file", "proportion_reads_duplicated"),
        ("file", "total_reads"),
        ("file", "average_base_quality"),
    ],
    "read_group_qc_probes": [
        ("read_group_qc", "total_sequences"),
        ("read_group_qc", "percent_gc_content"),
        ("read_group_qc", "basic_statistics"),
    ],
}

rna_d_field_rows = []
for file_id, node_map in rna_d_nodes.items():
    identity = rna_d_identity.loc[
        rna_d_identity["file_id"].eq(file_id)
    ].iloc[0]
    linked = bool(
        identity["n_exact_hierarchy_paths"] > 0
        and identity["live_direct_association"] == "EXACT_MATCH"
    )
    for domain, probes in RNA_D_PROBES.items():
        for scope, field in probes:
            nodes = node_map[scope] if (
                linked or scope in ("file", "read_group_qc")
            ) else []
            present, informative, discordant = source_value_status(nodes, field)
            # Structured FastQC summaries are reported as structural presence;
            # their contents are NOT converted to quantitative QC scores.
            structured = any(
                isinstance(node.get(field), (dict, list))
                for node in nodes if isinstance(node, dict)
            )
            values = sorted({
                str(node[field]).strip()
                for node in nodes if isinstance(node, dict)
                and field in node and informative_scalar(node[field])
            })
            rna_d_field_rows.append({
                "file_id": file_id,
                "case_uuid": identity["case_uuid"],
                "sample_uuid": identity["sample_uuid"],
                "aliquot_uuid": identity["aliquot_uuid"],
                "primary_site": identity["primary_site"],
                "domain": domain,
                "scope": scope,
                "field": field,
                "biospecimen_link_eligible_for_evidence": linked,
                "key_present": present,
                "informative_scalar": informative,
                "informative_values_json": json.dumps(values, ensure_ascii=False),
                "structured_value_present": structured,
                "multiple_informative_values_within_file": discordant,
                "interpretation": (
                    "AVAILABILITY_ONLY_NOT_A_QC_THRESHOLD"
                    if scope in ("file", "read_group_qc")
                    else "EXACT_LINKED_BIOSPECIMEN_METADATA_ONLY"
                ),
            })
rna_d_field_evidence = pd.DataFrame(rna_d_field_rows).sort_values(
    ["case_uuid", "file_id", "domain", "field"], kind="stable"
).reset_index(drop=True)
display(rna_d_field_evidence.groupby(["domain", "field"]).agg(
    n_files_with_informative_scalar=("informative_scalar", "sum"),
    n_files_with_structured_field=("structured_value_present", "sum"),
    n_files_with_key=("key_present", "sum"),
).reset_index())
print("No field value interpreted as sufficient for a selection rule")

,domain,field,n_files_with_informative_scalar,n_files_with_structured_field,n_files_with_key
0,collection_provenance,days_to_collection,0,0,12
1,collection_provenance,days_to_sample_procurement,29,0,29
2,collection_provenance,freezing_method,29,0,29
3,collection_provenance,method_of_sample_procurement,29,0,29
4,collection_provenance,preservation_method,29,0,29
5,collection_provenance,time_between_excision_and_freezing,15,0,21
6,file_level_qc_probes,average_base_quality,0,0,0
7,file_level_qc_probes,proportion_reads_duplicated,0,0,0
8,file_level_qc_probes,proportion_reads_mapped,0,0,0
9,file_level_qc_probes,total_reads,0,0,0


No field value interpreted as sufficient for a selection rule


In [32]:
# =============================================================================
# Quantify whether available attributes differ across files within each case
# =============================================================================

rna_d_case_field_comparison = (
    rna_d_field_evidence.groupby(["case_uuid", "domain", "field"])
    .agg(
        n_files_with_key=("key_present", "sum"),
        n_files_informative=("informative_scalar", "sum"),
        n_files_structured=("structured_value_present", "sum"),
        n_files_without_exact_link=(
            "biospecimen_link_eligible_for_evidence",
            lambda values: int((~values).sum())
        ),
        n_distinct_informative_values=(
            "informative_values_json",
            lambda values: len({
                item for raw in values for item in json.loads(raw)
            }),
        ),
    ).reset_index()
)
rna_d_case_field_comparison["candidate_values_differ"] = (
    rna_d_case_field_comparison["n_distinct_informative_values"].gt(1)
)
display(
    rna_d_case_field_comparison.groupby("domain")
    .agg(
        n_case_fields_informative=(
            "n_files_informative", lambda v: int(v.gt(0).sum())
        ),
        n_case_fields_with_distinct_values=("candidate_values_differ", "sum"),
    ).reset_index()
)

rna_d_case_summary = (
    rna_d_pilot_cases[[
        "case_uuid", "primary_site_values", "rna_multiplicity_class",
        "n_files", "n_samples", "n_aliquots",
    ]].merge(
        rna_d_identity.groupby("case_uuid")
        .agg(
            n_files_with_matching_file_identity=("live_file_identity", "size"),
            n_files_with_exact_biospecimen_link=(
                "n_exact_hierarchy_paths",
                lambda s: int(s.gt(0).sum()),
            ),
            n_files_with_nested_path=(
                "n_exact_hierarchy_paths", lambda s: int(s.gt(0).sum())
            ),
            n_files_with_read_group_qc=(
                "n_read_group_qcs", lambda s: int(s.gt(0).sum())
            ),
        ).reset_index(),
        on="case_uuid", validate="one_to_one"
    )
)
for domain in RNA_D_PROBES:
    coverage = (
        rna_d_field_evidence.loc[
            rna_d_field_evidence["domain"].eq(domain)
            & rna_d_field_evidence["informative_scalar"]
        ]
        .groupby("case_uuid")["file_id"].nunique()
    )
    rna_d_case_summary["files_informative_" + domain] = (
        rna_d_case_summary["case_uuid"].map(coverage).fillna(0).astype("int64")
    )
rna_d_case_summary["selection_status"] = "NOT_FROZEN"
display(rna_d_case_summary.groupby(
    ["rna_multiplicity_class", "primary_site_values"]
).agg(
    n_cases=("case_uuid", "nunique"),
    n_files=("n_files", "sum"),
    exact_nested_path_files=("n_files_with_nested_path", "sum"),
    read_group_qc_files=("n_files_with_read_group_qc", "sum"),
    files_with_spatial_specimen=("files_informative_spatial_specimen", "sum"),
    files_with_collection_provenance=(
        "files_informative_collection_provenance", "sum"
    ),
    files_with_molecular_preparation=(
        "files_informative_molecular_preparation", "sum"
    ),
    files_with_file_qc_probes=(
        "files_informative_file_level_qc_probes", "sum"
    ),
).reset_index())
print("Full 649-file recovery feasibility: NOT ASSESSED — pilot only")

,domain,n_case_fields_informative,n_case_fields_with_distinct_values
0,collection_provenance,57,0
1,file_level_qc_probes,0,0
2,molecular_preparation,0,0
3,read_group_qc_probes,0,0
4,spatial_specimen,29,0


,rna_multiplicity_class,primary_site_values,n_cases,n_files,exact_nested_path_files,read_group_qc_files,files_with_spatial_specimen,files_with_collection_provenance,files_with_molecular_preparation,files_with_file_qc_probes
0,MULTIPLE_SAMPLES_IN_CASE,Kidney,3,9,9,0,9,9,0,0
1,MULTIPLE_SAMPLES_IN_CASE,Larynx,3,6,6,0,6,6,0,0
2,MULTIPLE_SAMPLES_IN_CASE,"Uterus, NOS",3,6,6,0,6,6,0,0
3,ONE_SAMPLE_MULTIPLE_ALIQUOTS,Pancreas,3,8,8,0,8,8,0,0


Full 649-file recovery feasibility: NOT ASSESSED — pilot only


In [33]:
# =============================================================================
# Preserve no-adjudication and precise scope before writing pilot diagnostics
# =============================================================================

if (
    len(rna_d_case_summary) != 12
    or rna_d_case_summary["case_uuid"].duplicated().any()
    or len(rna_d_identity) != len(rna_d_file_ids)
    or len(rna_d_field_evidence) != (
        len(rna_d_file_ids) * sum(len(p) for p in RNA_D_PROBES.values())
    )
    or not set(rna_d_identity["file_id"]).issubset(
        set(rna_adjudication_files["file_id"])
    )
    or rna_d_case_summary["selection_status"].ne("NOT_FROZEN").any()
    or rna_adjudication_files["candidate_selected"].any()
    or rna_adjudication_cases["selected_file_id"].notna().any()
):
    raise ValueError("PREFLIGHT-D violated exact scope or no-selection boundary")

print("PREFLIGHT-D pilot diagnostic ready for review; no automatic expansion")
print("All 288 multiplicity cases remain unresolved, no biological scoring")

PREFLIGHT-D pilot diagnostic ready for review; no automatic expansion
All 288 multiplicity cases remain unresolved, no biological scoring


## PREFLIGHT-E — Provisional single-file RNA technical readiness and lineage attrition

**Analysis unit is the case, not the GDC file.** The 1,082 \`SINGLE_FILE_SAMPLE_ALIQUOT\` cases have one **provisional primary RNA-seq file** each, but are not yet an eligible or independent external cohort. A case can have *additional* GDC RNA files outside that primary subset, including unresolved ancestry; capture such cases explicitly rather than declaring their entire case history uniquely sampled.

**Prospectively bounded technical work:** inspect the \`unstranded\` count vector and \`gene_id\` column of each of these 1,082 already downloaded STAR Counts files **once**. Reuse Notebook 111's frozen file identity and reported full MD5 PASS; do not rehash, download, normalize, score, model, select thresholds or filter genes. Record presence of the exact 2,389 frozen Ensembl IDs and descriptive count properties. Track the STAR special \`N_\` records separately: they are **not genes**, and their read counts are **not an independent alignment QC report**. Distinguish absent gene identifiers from measured zero counts and never silently strip Ensembl versions to rescue a match.

**Quality limits:** library-size/detected-feature statistics and STAR summary counts are *technical diagnostics*, not comparative read quality, tumor purity, batch correction, or proof of sample equivalence. Missing alignment logs/RIN/FASTQ-derived QC remain unresolved. No sample is called PASS/FAIL using observed empirical distributions; all numerical thresholds and lineage minima require later prospective review.

**Loss/selection-bias boundary:** quantify by unharmonized provider \`primary_site\` × \`disease_type\` how many *distinct cases* are provisional primary, structurally single-file or in the unresolved 288. Flag any other same-case RNA files across the full Notebook 111 index, and do not treat overlapping ambiguous ancestry as a new independent denominator. These descriptive labels are not the final TCGA lineage crosswalk.

**End state:** non-frozen file-level technical inventory, exact-gene coverage audit, and source-stratum attrition table. Neither the 1,082 cases nor the 288 multiple-file cases are adjudicated. Cohort eligibility, external donor independence, preprocessing, matched nulls and biological replication remain **NOT FROZEN**.

In [34]:
# =============================================================================
# Define the 1,082 unique-primary-file candidate scope from 800-A only
# =============================================================================

rna_e_simple_cases = rna_cases.loc[
    rna_cases["rna_multiplicity_class"].eq("SINGLE_FILE_SAMPLE_ALIQUOT"),
    ["case_uuid", "primary_site_values", "disease_type_values",
     "n_files", "n_samples", "n_aliquots"],
].copy()

rna_e_simple_files = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq")
    & candidate_primary["case_uuid"].isin(rna_e_simple_cases["case_uuid"]),
    ["case_uuid", "file_id", "filename", "sample_uuid", "aliquot_uuid",
     "primary_site", "disease_type", "size_bytes", "provider_md5"],
].copy().sort_values(["case_uuid", "file_id"], kind="stable").reset_index(drop=True)

if (
    len(rna_e_simple_cases) != 1082
    or len(rna_e_simple_files) != len(rna_e_simple_cases)
    or rna_e_simple_files["case_uuid"].duplicated().any()
    or rna_e_simple_files["file_id"].duplicated().any()
    or not rna_e_simple_cases["n_files"].eq(1).all()
    or not rna_e_simple_cases["n_samples"].eq(1).all()
    or not rna_e_simple_cases["n_aliquots"].eq(1).all()
    or set(rna_e_simple_files["case_uuid"]) != set(rna_e_simple_cases["case_uuid"])
):
    raise ValueError("PREFLIGHT-E simple-case universe contradicts 800-A")

# Same-case RNA files outside provisional primary count must remain visible.
rna_e_all_source = file_index.loc[file_index["modality"].eq("rnaseq")]
rna_e_full_case_counts = (
    rna_e_all_source.groupby("case_uuid", dropna=False)
    .agg(
        n_all_rna_files_in_case=("file_id", "nunique"),
        n_unresolved_ancestry_rna_files=(
            "mapping_status",
            lambda v: int(v.eq("ambiguous_sample_ancestry").sum())
        ),
    ).reset_index()
)
rna_e_simple_files = rna_e_simple_files.merge(
    rna_e_full_case_counts, on="case_uuid", how="left", validate="one_to_one"
)
if rna_e_simple_files["n_all_rna_files_in_case"].isna().any():
    raise ValueError("A single-file primary case is absent from 111 source inventory")
rna_e_simple_files["n_other_rna_files_in_case"] = (
    rna_e_simple_files["n_all_rna_files_in_case"].astype("int64") - 1
)
if rna_e_simple_files["n_other_rna_files_in_case"].lt(0).any():
    raise ValueError("Source inventory has fewer RNA files than primary candidates")

print("Single provisional-primary RNA file/case:", len(rna_e_simple_files))
print("Cases with other same-case RNA files (any source category):",
      int(rna_e_simple_files["n_other_rna_files_in_case"].gt(0).sum()))
print("Cases with at least one unresolved-ancestry RNA file:",
      int(rna_e_simple_files["n_unresolved_ancestry_rna_files"].gt(0).sum()))
print("No case declared fully unique, clinically independent or eligible")

Single provisional-primary RNA file/case: 1082
Cases with other same-case RNA files (any source category): 530
Cases with at least one unresolved-ancestry RNA file: 37
No case declared fully unique, clinically independent or eligible


In [35]:
# =============================================================================
# Inventory frozen Ensembl identifiers and prespecify purely technical fields
# =============================================================================

rna_e_frozen_ids = shared_genes["tumor_gene_id"].astype("string")
if (
    len(rna_e_frozen_ids) != 2389
    or rna_e_frozen_ids.isna().any()
    or rna_e_frozen_ids.duplicated().any()
):
    raise ValueError("Unexpected frozen tumor Ensembl gene ID universe")
rna_e_frozen_set = set(rna_e_frozen_ids)
rna_e_frozen_bases = rna_e_frozen_ids.str.replace(
    r"\.\d+$", "", regex=True
)
if rna_e_frozen_bases.duplicated().any():
    raise ValueError("Duplicate version-insensitive frozen IDs require review")

RNA_E_COUNT_COLUMN = "unstranded"  # inspection only; no use in scoring
RNA_E_STAR_UNASSIGNED = (
    "N_unmapped", "N_multimapping", "N_noFeature", "N_ambiguous",
)
RNA_E_STAR_ROOT = (
    Paths.root / "data" / "raw" / "cptac3" / "rnaseq_star_counts"
)
rna_e_frozen_file_presence = np.zeros(len(rna_e_frozen_ids), dtype=np.int64)
print("Prespecified exact frozen IDs:", len(rna_e_frozen_ids))
print("STAR column (diagnostic only):", RNA_E_COUNT_COLUMN)
print("No coverage cutoff, library-size cutoff or sample QC pass/fail threshold")

Prespecified exact frozen IDs: 2389
STAR column (diagnostic only): unstranded
No coverage cutoff, library-size cutoff or sample QC pass/fail threshold


In [36]:
# =============================================================================
# Single-pass local STAR file diagnostics; read no unrequested count measures
# =============================================================================

def rna_e_inspect_star_counts(record):
    path = RNA_E_STAR_ROOT / str(record.file_id) / str(record.filename)
    if not path.is_file():
        raise FileNotFoundError(
            "Previously MD5-audited source payload missing: " + str(path)
        )
    if path.stat().st_size != int(record.size_bytes):
        raise ValueError("Provider size differs at frozen file: " + str(record.file_id))

    table = pd.read_csv(
        path, sep="\t", comment="#",
        usecols=["gene_id", RNA_E_COUNT_COLUMN],
        dtype={"gene_id": "string", RNA_E_COUNT_COLUMN: "string"},
    )
    if table["gene_id"].isna().any():
        raise ValueError("Missing STAR gene identifiers: " + str(record.file_id))

    special_mask = table["gene_id"].str.startswith("N_", na=False)
    special = table.loc[special_mask]
    gene_rows = table.loc[~special_mask]
    gene_ids = gene_rows["gene_id"]
    if gene_ids.duplicated().any():
        raise ValueError("Duplicate exact STAR gene ID: " + str(record.file_id))
    if gene_ids.empty:
        raise ValueError("No STAR gene rows: " + str(record.file_id))

    numeric = pd.to_numeric(gene_rows[RNA_E_COUNT_COLUMN], errors="coerce")
    counts = numeric.to_numpy(dtype="float64", na_value=np.nan)
    numeric_ok = (
        bool(np.isfinite(counts).all())
        and bool((counts >= 0).all())
        and bool((counts == np.floor(counts)).all())
    )
    # Diagnostic counts are recorded only when the unstranded vector is valid.
    exact_presence = rna_e_frozen_ids.isin(set(gene_ids))
    base_ids = gene_ids.str.replace(r"\.\d+$", "", regex=True)
    base_counts = base_ids.value_counts(dropna=False)
    frozen_base_match = rna_e_frozen_bases.isin(set(base_ids))
    exact_n = int(exact_presence.sum())

    summary = {}
    special_numeric = pd.to_numeric(
        special[RNA_E_COUNT_COLUMN], errors="coerce"
    )
    for special_id in RNA_E_STAR_UNASSIGNED:
        observed = special_numeric.loc[special["gene_id"].eq(special_id)]
        summary[special_id] = (
            float(observed.iloc[0]) if len(observed) == 1
            and pd.notna(observed.iloc[0]) else np.nan
        )

    return {
        "n_gene_rows": len(gene_rows),
        "n_duplicate_versionless_ids": int(base_counts.gt(1).sum()),
        "n_star_special_rows": len(special),
        "n_frozen_exact_present": exact_n,
        "n_frozen_exact_missing": len(rna_e_frozen_ids) - exact_n,
        "n_frozen_versionless_diagnostic": int(frozen_base_match.sum()),
        "n_frozen_versionless_only_diagnostic": int(
            (frozen_base_match & ~exact_presence).sum()
        ),
        "unstranded_nonnegative_integer_vector": numeric_ok,
        "n_invalid_unstranded_gene_values": int(
            (~np.isfinite(counts) | (counts < 0) |
             (np.isfinite(counts) & (counts != np.floor(counts)))).sum()
        ),
        "unstranded_gene_count_total": (
            float(counts.sum()) if numeric_ok else np.nan
        ),
        "n_detected_star_genes": (
            int((counts > 0).sum()) if numeric_ok else pd.NA
        ),
        "n_detected_frozen_genes_exact": (
            int((counts[gene_ids.isin(rna_e_frozen_set).to_numpy()] > 0).sum())
            if numeric_ok else pd.NA
        ),
        **{"star_" + key.lower(): summary[key] for key in RNA_E_STAR_UNASSIGNED},
    }, exact_presence.to_numpy(dtype="int64")

rna_e_scan_records = []
for index, record in enumerate(
    rna_e_simple_files.itertuples(index=False), start=1
):
    diagnostic, presence_vector = rna_e_inspect_star_counts(record)
    rna_e_frozen_file_presence += presence_vector
    rna_e_scan_records.append({
        "case_uuid": record.case_uuid,
        "file_id": record.file_id,
        "sample_uuid": record.sample_uuid,
        "aliquot_uuid": record.aliquot_uuid,
        **diagnostic,
    })
    if index % 200 == 0 or index == len(rna_e_simple_files):
        print("PREFLIGHT-E STAR count feature reads:",
              index, "/", len(rna_e_simple_files))

rna_e_star_inventory = pd.DataFrame(rna_e_scan_records)
print("STAR feature/count technical inventory completed:",
      len(rna_e_star_inventory))
print("File MD5s were NOT recomputed; Notebook 111 full MD5 PASS reused")

PREFLIGHT-E STAR count feature reads: 200 / 1082
PREFLIGHT-E STAR count feature reads: 400 / 1082
PREFLIGHT-E STAR count feature reads: 600 / 1082
PREFLIGHT-E STAR count feature reads: 800 / 1082
PREFLIGHT-E STAR count feature reads: 1000 / 1082
PREFLIGHT-E STAR count feature reads: 1082 / 1082
STAR feature/count technical inventory completed: 1082
File MD5s were NOT recomputed; Notebook 111 full MD5 PASS reused


In [37]:
# =============================================================================
# Reconcile technical readouts, preserving failures as non-frozen diagnostics
# =============================================================================

rna_e_file_diagnostics = (
    rna_e_simple_files.merge(
        rna_e_star_inventory, on=["case_uuid", "file_id", "sample_uuid",
                                   "aliquot_uuid"], validate="one_to_one"
    ).sort_values(["case_uuid", "file_id"], kind="stable").reset_index(drop=True)
)
rna_e_file_diagnostics["exact_frozen_feature_coverage_fraction"] = (
    rna_e_file_diagnostics["n_frozen_exact_present"] / len(rna_e_frozen_ids)
)
rna_e_file_diagnostics["structural_scope_status"] = (
    "SINGLE_PROVISIONAL_PRIMARY_FILE_NOT_ELIGIBILITY"
)
rna_e_file_diagnostics["technical_qc_interpretation"] = (
    "DESCRIPTIVE_STAR_COUNTS_ONLY_NOT_SEQUENCING_QC"
)
rna_e_file_diagnostics["selection_performed"] = False

rna_e_gene_coverage = shared_genes[["gene_symbol", "tumor_gene_id"]].copy()
rna_e_gene_coverage["n_simple_files_exact_id_present"] = (
    rna_e_frozen_file_presence
)
rna_e_gene_coverage["n_simple_files_inspected"] = len(rna_e_file_diagnostics)
rna_e_gene_coverage["fraction_simple_files_exact_id_present"] = (
    rna_e_gene_coverage["n_simple_files_exact_id_present"]
    / len(rna_e_file_diagnostics)
)

if (
    len(rna_e_file_diagnostics) != len(rna_e_simple_files)
    or rna_e_file_diagnostics["file_id"].duplicated().any()
    or rna_e_file_diagnostics["selection_performed"].any()
    or not set(rna_e_file_diagnostics["file_id"]).isdisjoint(
        set(rna_adjudication_files["file_id"])
    )
    or int(rna_e_file_diagnostics["n_frozen_exact_present"].sum()) != int(
        rna_e_gene_coverage["n_simple_files_exact_id_present"].sum()
    )
):
    raise ValueError("PREFLIGHT-E gene coverage and file identity do not reconcile")

display(rna_e_file_diagnostics[[
    "n_gene_rows", "n_frozen_exact_present", "n_frozen_exact_missing",
    "n_frozen_versionless_only_diagnostic",
    "unstranded_gene_count_total", "n_detected_star_genes",
]].describe().T)
print("Files with incomplete exact frozen gene-ID coverage:",
      int(rna_e_file_diagnostics["n_frozen_exact_missing"].gt(0).sum()))
print("Files with non-integer/negative/nonfinite STAR count values:",
      int((~rna_e_file_diagnostics[
          "unstranded_nonnegative_integer_vector"].astype(bool)
      ).sum()))
print("No file excluded; observed distributions cannot set QC thresholds")

,count,mean,std,min,25%,50%,75%,max
n_gene_rows,1082.0,6.066000e+04,0.000000e+00,60660.0,60660.0,60660.0,60660.00,60660.0
n_frozen_exact_present,1082.0,2.389000e+03,0.000000e+00,2389.0,2389.0,2389.0,2389.00,2389.0
n_frozen_exact_missing,1082.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.00,0.0
n_frozen_versionless_only_diagnostic,1082.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.00,0.0
unstranded_gene_count_total,1082.0,3.883843e+07,8.633513e+06,1074810.0,33986743.5,38347562.0,43393965.25,84641397.0
n_detected_star_genes,1082.0,4.205390e+04,2.025331e+03,20953.0,41077.5,42264.0,43245.75,46782.0


Files with incomplete exact frozen gene-ID coverage: 0
Files with non-integer/negative/nonfinite STAR count values: 0
No file excluded; observed distributions cannot set QC thresholds


In [38]:
# =============================================================================
# Quantify primary-case structural attrition by UNHARMONIZED source tumor labels
# =============================================================================

rna_e_total_case_context = rna_cases[[
    "case_uuid", "primary_site_values", "disease_type_values",
    "rna_multiplicity_class",
]].copy()
rna_e_total_case_context["single_primary_file"] = (
    rna_e_total_case_context["rna_multiplicity_class"].eq(
        "SINGLE_FILE_SAMPLE_ALIQUOT"
    )
)
rna_e_technical_case_status = rna_e_file_diagnostics[[
    "case_uuid", "n_frozen_exact_missing",
    "unstranded_nonnegative_integer_vector",
    "n_other_rna_files_in_case", "n_unresolved_ancestry_rna_files",
]].copy()
rna_e_total_case_context = rna_e_total_case_context.merge(
    rna_e_technical_case_status, on="case_uuid", how="left",
    validate="one_to_one"
)
rna_e_total_case_context["simple_count_vector_diagnostic_complete"] = (
    rna_e_total_case_context["single_primary_file"]
    & rna_e_total_case_context["unstranded_nonnegative_integer_vector"].fillna(False)
)
rna_e_total_case_context["simple_all_exact_frozen_ids_present"] = (
    rna_e_total_case_context["single_primary_file"]
    & rna_e_total_case_context["n_frozen_exact_missing"].eq(0).fillna(False)
)
rna_e_total_case_context["simple_case_other_rna_source_files"] = (
    rna_e_total_case_context["single_primary_file"]
    & rna_e_total_case_context["n_other_rna_files_in_case"].gt(0).fillna(False)
)
rna_e_total_case_context["simple_case_ambiguous_rna_source_files"] = (
    rna_e_total_case_context["single_primary_file"]
    & rna_e_total_case_context[
        "n_unresolved_ancestry_rna_files"].gt(0).fillna(False)
)
rna_e_source_attrition = (
    rna_e_total_case_context.groupby(
        ["primary_site_values", "disease_type_values"], dropna=False
    ).agg(
        n_provisional_primary_cases=("case_uuid", "nunique"),
        n_single_primary_file_cases=("single_primary_file", "sum"),
        n_single_complete_count_vector=(
            "simple_count_vector_diagnostic_complete", "sum"
        ),
        n_single_all_exact_frozen_ids=(
            "simple_all_exact_frozen_ids_present", "sum"
        ),
        n_single_with_other_source_rna_files=(
            "simple_case_other_rna_source_files", "sum"
        ),
        n_single_with_ambiguous_source_rna_files=(
            "simple_case_ambiguous_rna_source_files", "sum"
        ),
    ).reset_index()
)
rna_e_source_attrition["n_multiplicity_cases_unadjudicated"] = (
    rna_e_source_attrition["n_provisional_primary_cases"]
    - rna_e_source_attrition["n_single_primary_file_cases"]
)
rna_e_source_attrition["fraction_cases_structurally_simple"] = (
    rna_e_source_attrition["n_single_primary_file_cases"]
    / rna_e_source_attrition["n_provisional_primary_cases"]
)
rna_e_source_attrition = rna_e_source_attrition.sort_values(
    ["n_provisional_primary_cases", "primary_site_values"],
    ascending=[False, True], kind="stable"
).reset_index(drop=True)

if (
    rna_e_source_attrition["n_provisional_primary_cases"].sum() != len(rna_cases)
    or rna_e_source_attrition["n_single_primary_file_cases"].sum()
    != len(rna_e_file_diagnostics)
    or rna_e_source_attrition["n_multiplicity_cases_unadjudicated"].sum()
    != len(rna_adjudication_cases)
):
    raise ValueError("PREFLIGHT-E source-context attrition did not reconcile")

display(rna_e_source_attrition.head(25))
print("This table is provider-label stratification, NOT a TCGA lineage crosswalk")
print("No lineage minimum, QC cutoff or cohort exclusion was applied")

,primary_site_values,disease_type_values,n_provisional_primary_cases,n_single_primary_file_cases,n_single_complete_count_vector,n_single_all_exact_frozen_ids,n_single_with_other_source_rna_files,n_single_with_ambiguous_source_rna_files,n_multiplicity_cases_unadjudicated,fraction_cases_structurally_simple
0,"Uterus, NOS",Adenomas and Adenocarcinomas,235,56,56,56,20,15,179,0.238298
1,Kidney,Adenomas and Adenocarcinomas,232,192,192,192,116,2,40,0.827586
2,Bronchus and lung,Adenomas and Adenocarcinomas,178,178,178,178,168,9,0,1.000000
3,Brain,Gliomas,159,159,159,159,17,0,0,1.000000
4,Stomach,Adenomas and Adenocarcinomas,142,142,142,142,27,0,0,1.000000
5,Skin,Nevi and Melanomas,138,138,138,138,55,0,0,1.000000
6,Pancreas,Ductal and Lobular Neoplasms,100,88,88,88,26,0,12,0.880000
7,Bronchus and lung,Squamous Cell Neoplasms,90,89,89,89,78,2,1,0.988889
8,Larynx,Squamous Cell Neoplasms,37,18,18,18,11,3,19,0.486486
9,Other and unspecified parts of tongue,Squamous Cell Neoplasms,19,7,7,7,5,3,12,0.368421


This table is provider-label stratification, NOT a TCGA lineage crosswalk
No lineage minimum, QC cutoff or cohort exclusion was applied


In [39]:
# =============================================================================
# Non-selection gate: technical findings are a diagnostic inventory, not a freeze
# =============================================================================

if (
    len(rna_e_file_diagnostics) != 1082
    or len(rna_adjudication_cases) != 288
    or rna_e_file_diagnostics["selection_performed"].any()
    or rna_adjudication_files["candidate_selected"].any()
    or rna_adjudication_cases["selected_file_id"].notna().any()
    or not case_multiplicity["selected_file_id"].isna().all()
    or not case_multiplicity["selection_status"].eq("NOT_FROZEN").all()
):
    raise ValueError("PREFLIGHT-E mutated the upstream no-selection boundary")
print("PREFLIGHT-E descriptive source technical inventory: READY TO PERSIST")
print("Simple cases:", len(rna_e_file_diagnostics),
      "| unresolved multiplicity cases:", len(rna_adjudication_cases))
print("Full read-quality QC, matched donors, TCGA lineage crosswalk: OPEN")
print("Analytic eligibility / normalization / program scoring: NOT AUTHORIZED")

PREFLIGHT-E descriptive source technical inventory: READY TO PERSIST
Simple cases: 1082 | unresolved multiplicity cases: 288
Full read-quality QC, matched donors, TCGA lineage crosswalk: OPEN
Analytic eligibility / normalization / program scoring: NOT AUTHORIZED


## Lexical identifier-overlap audit

Compare exact GDC case UUID and submitter strings against the frozen TCGA cohort. This is a **collision screen only**. Separate projects may assign different IDs to the same biological donor; therefore an absence of collisions is not evidence of documented donor independence. A provenance/crosswalk review remains a gate before confirmatory replication claims.

In [40]:
# =============================================================================
# Read TCGA source identities from the frozen Phase 2 sample mapping
# =============================================================================

rna_primary = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq")
].copy()

tcga_mapping = pd.read_csv(frozen_paths["tcga_sample_mapping"], dtype="string")
tcga_required = {"rna_case_uuid", "case_submitter_id"}
if tcga_required - set(tcga_mapping.columns):
    raise KeyError("Frozen TCGA mapping lacks required case identifiers")

tcga_case_uuid_set = set(tcga_mapping["rna_case_uuid"].dropna())
tcga_submitter_set = set(tcga_mapping["case_submitter_id"].dropna())

print("Candidate CPTAC-3 RNA cases:", rna_primary["case_uuid"].nunique())
print("Donor independence: UNRESOLVED without source crosswalk")

Candidate CPTAC-3 RNA cases: 1370
Donor independence: UNRESOLVED without source crosswalk


In [41]:
# =============================================================================
# Create a case-level exact-string collision screen, not independence labels
# =============================================================================

identifier_collisions = (
    candidate_primary.groupby(["modality", "case_uuid"], dropna=False)
    .agg(
        n_submitter_ids=("case_submitter_id", "nunique"),
        submitter_id_values=(
            "case_submitter_id",
            lambda v: "|".join(sorted(set(v.dropna()))),
        ),
        primary_site_values=(
            "primary_site", lambda v: "|".join(sorted(set(v.dropna())))
        ),
        disease_type_values=(
            "disease_type", lambda v: "|".join(sorted(set(v.dropna())))
        ),
    )
    .reset_index()
)
identifier_collisions["tcga_case_uuid_collision"] = (
    identifier_collisions["case_uuid"].isin(tcga_case_uuid_set)
)
identifier_collisions["tcga_submitter_collision"] = (
    identifier_collisions["submitter_id_values"].map(
        lambda s: any(v in tcga_submitter_set for v in s.split("|") if v)
    )
)
identifier_collisions["donor_independence_status"] = (
    "UNRESOLVED_WITHOUT_EXTERNAL_PROVENANCE"
)
identifier_collisions["case_submitter_ambiguity"] = (
    identifier_collisions["n_submitter_ids"].gt(1)
)

display(
    identifier_collisions.groupby("modality")
    .agg(
        candidate_cases=("case_uuid", "nunique"),
        exact_uuid_collision_rows=("tcga_case_uuid_collision", "sum"),
        exact_submitter_collision_rows=("tcga_submitter_collision", "sum"),
    )
)

,candidate_cases,exact_uuid_collision_rows,exact_submitter_collision_rows
modality,,,
methylation,1336,0,0
rnaseq,1370,0,0


## Pilot technical gene-ID compatibility

The source library has already passed Notebook 111 file-integrity checks. A single deterministically selected *provisional primary RNA-seq* file is opened here **only** to inspect the GDC STAR Counts header and Ensembl identifier mapping to the 2,389-gene frozen universe.

This is a technical pilot, **not** a complete feature-coverage assessment, cohort normalization, or program test. A match after stripping Ensembl versions is reported only as a diagnostic; no version stripping, symbol fallback, missing-gene imputation or rescue is applied. Full per-file/lineage compatibility and thresholds remain open until reviewed.

In [42]:
# =============================================================================
# Select one deterministic RNA-seq technical pilot from the frozen source files
# =============================================================================

if rna_primary.empty:
    raise ValueError("Notebook 111 has no provisional primary RNA-seq candidates")
pilot_record = rna_primary.sort_values(
    ["file_id", "filename"], kind="stable"
).iloc[0]
pilot_file_uuid = pilot_record["file_id"]
pilot_path = (
    Paths.root / "data" / "raw" / "cptac3" / "rnaseq_star_counts"
    / pilot_file_uuid / pilot_record["filename"]
)
if not pilot_path.is_file():
    raise FileNotFoundError(f"Frozen Notebook 111 pilot payload is missing: {pilot_path}")

pilot_header = pd.read_csv(pilot_path, sep="\t", comment="#", nrows=0).columns.tolist()
print("Pilot GDC file:", pilot_file_uuid)
print("STAR Counts fields:", pilot_header)
if not {"gene_id", "unstranded"}.issubset(pilot_header):
    raise ValueError("Unexpected STAR Counts schema; no gene crosswalk attempted")

Pilot GDC file: 006f3179-76c6-4525-9560-2cf2a3e3e88f
STAR Counts fields: ['gene_id', 'gene_name', 'gene_type', 'unstranded', 'stranded_first', 'stranded_second', 'tpm_unstranded', 'fpkm_unstranded', 'fpkm_uq_unstranded']


In [43]:
# =============================================================================
# Audit exact/version-insensitive Ensembl ID coverage in the technical pilot
# =============================================================================

pilot_gene_ids = pd.read_csv(
    pilot_path, sep="\t", comment="#", usecols=["gene_id"], dtype="string"
)["gene_id"].dropna()
provider_ids = set(pilot_gene_ids.tolist())
provider_versions_stripped = pilot_gene_ids.str.replace(
    r"\.\d+$", "", regex=True
)
base_id_counts = provider_versions_stripped.value_counts()
ambiguous_base_ids = set(base_id_counts[base_id_counts.gt(1)].index)

gene_id_pilot = shared_genes[
    ["gene_symbol", "tumor_gene_id", "cell_line_gene_id"]
].copy()
gene_id_pilot["pilot_file_id"] = pilot_file_uuid
gene_id_pilot["exact_ensembl_match"] = (
    gene_id_pilot["tumor_gene_id"].isin(provider_ids)
)
gene_id_pilot["versionless_pilot_match_diagnostic"] = (
    gene_id_pilot["tumor_gene_id"].str.replace(r"\.\d+$", "", regex=True)
    .isin(set(provider_versions_stripped))
)
gene_id_pilot["versionless_pilot_id_ambiguous"] = (
    gene_id_pilot["tumor_gene_id"].str.replace(r"\.\d+$", "", regex=True)
    .isin(ambiguous_base_ids)
)
gene_id_pilot["mapping_applied"] = False

display(gene_id_pilot[
    ["exact_ensembl_match", "versionless_pilot_match_diagnostic",
     "versionless_pilot_id_ambiguous"]
].sum().to_frame(name="genes"))
print("Cohort-wide feature compatibility: NOT ASSESSED")

,genes
exact_ensembl_match,2389
versionless_pilot_match_diagnostic,2389
versionless_pilot_id_ambiguous,0


Cohort-wide feature compatibility: NOT ASSESSED


## Readiness and persistence

The preflight outputs remain local, provisional diagnostics under `data/interim/metadata`. They must not be registered as frozen biological validation evidence or used for Phase 9. The RNA-seq per-case selection and lineage crosswalk remain unresolved; project-specific donor independence remains unverified; the statistical Phase 8 analysis contract is not frozen.

New outputs receive basic shape/column validation after persistence, while Notebook 111's prior raw MD5 scan, upstream source inventories and frozen program-construction logic are not rerun.

In [44]:
# =============================================================================
# Assemble explicit readiness statuses and input provenance
# =============================================================================

input_provenance = {
    "111_summary": {
        "path": project_relative_path(AUDIT_SUMMARY_PATH),
        "sha256": calculate_sha256(AUDIT_SUMMARY_PATH),
    },
    "111_registry_proposal": {
        "path": project_relative_path(RAW_REGISTRY_PROPOSAL_PATH),
        "sha256": calculate_sha256(RAW_REGISTRY_PROPOSAL_PATH),
    },
}
input_provenance.update({
    "111_" + name: {
        "path": project_relative_path(path),
        "sha256": audit_summary["audit_artifacts"][name]["sha256"],
    }
    for name, path in audit_inputs.items()
})
input_provenance["gdc_targeted_rna_pilot_for_d"] = {
    "path": project_relative_path(RNA_D_SNAPSHOT_PATH),
    "sha256": calculate_sha256(RNA_D_SNAPSHOT_PATH),
    "query_sha256": rna_d_query_sha,
    "captured_at_utc": rna_d_snapshot.get("captured_at_utc"),
}
input_provenance["gdc_rna_acquisition_metadata_for_c"] = {
    "path": project_relative_path(RNA_GDC_METADATA_PATH),
    "sha256": expected_rna_metadata_sha,
}
input_provenance.update({
    artifact_id: {
        "path": get_artifact(artifact_registry, artifact_id)["path"],
        "sha256": get_artifact(artifact_registry, artifact_id)["sha256"],
    }
    for artifact_id in FROZEN_ARTIFACT_IDS.values()
})

readiness_status = {
    "notebook111_source_integrity": "UPSTREAM_PASS_REUSED",
    "notebook111_local_handoff_identity": "PASS",
    "phase4_frozen_identity": "PASS",
    "provisional_primary_inventory": "AVAILABLE_NOT_COHORT_FROZEN",
    "one_record_per_case": "PENDING_POLICY",
    "rna_multiplicity_adjudication_evidence": "DESCRIBED_UNRESOLVED_NO_SELECTION",
    "rna_specimen_provenance_metadata": "SNAPSHOT_AVAILABILITY_AUDITED",
    "rna_quantitative_qc_metadata": "SNAPSHOT_PROBES_ONLY_NO_PAYLOAD_QC",
    "rna_one_record_per_case_selection": "NOT_FROZEN",
    "rna_live_gdc_pilot": "TARGETED_SOURCE_SNAPSHOT_CAPTURED_NOT_COHORT_WIDE",
    "rna_d_additional_provenance_qc": (
        "PILOT_AVAILABILITY_REVIEW_PENDING"
        if rna_d_exact_link_status == "COMPLETE_EXACT_TRIPLET_RECOVERED"
        else "BLOCKED_EXACT_BIOSPECIMEN_LINK_INCOMPLETE"
    ),
    "rna_d_exact_biospecimen_link": rna_d_exact_link_status,
    "rna_case_multiplicity_classes": "DESCRIPTIVE_COMPLETE_NOT_RESOLVED",
    "rna_cross_file_ancestry_conflicts": (
        "REVIEW_REQUIRED" if rna_cases["cross_file_ancestry_conflict"].any()
        else "NONE_DETECTED_WITHIN_PROVISIONAL_PRIMARY_FILES"
    ),
    "harmonized_lineage_crosswalk": "NOT_ASSESSED",
    "donor_independence": "UNRESOLVED",
    "star_counts_gene_compatibility": "PREFLIGHT_E_SINGLE_PRIMARY_FULL_EXACT_ID_SCREEN",
    "full_transcriptomic_qc_and_coverage": (
        "SINGLE_PRIMARY_COUNT_FEATURE_DIAGNOSTICS_ONLY_NO_SEQUENCING_QC"
    ),
    "rna_e_single_primary_technical_inventory": "DESCRIPTIVE_COMPLETE_NOT_ELIGIBILITY",
    "rna_e_lineage_aware_attrition": "SOURCE_LABEL_STRATIFICATION_ONLY",
    "rna_e_qc_threshold_and_selection_policy": "NOT_FROZEN",
    "methylation_platform_and_sample_harmonization": "NOT_ASSESSED",
    "external_replication_analysis_contract": "NOT_FROZEN",
    "biological_validation_authorized": False,
}
display(pd.DataFrame(
    [{"gate": gate, "status": status} for gate, status in readiness_status.items()]
))

,gate,status
0,notebook111_source_integrity,UPSTREAM_PASS_REUSED
1,notebook111_local_handoff_identity,PASS
2,phase4_frozen_identity,PASS
3,provisional_primary_inventory,AVAILABLE_NOT_COHORT_FROZEN
4,one_record_per_case,PENDING_POLICY
5,rna_multiplicity_adjudication_evidence,DESCRIBED_UNRESOLVED_NO_SELECTION
6,rna_specimen_provenance_metadata,SNAPSHOT_AVAILABILITY_AUDITED
7,rna_quantitative_qc_metadata,SNAPSHOT_PROBES_ONLY_NO_PAYLOAD_QC
8,rna_one_record_per_case_selection,NOT_FROZEN
9,rna_live_gdc_pilot,TARGETED_SOURCE_SNAPSHOT_CAPTURED_NOT_COHORT_WIDE


In [45]:
# =============================================================================
# Persist new technical inventories; leave registries and raw data untouched
# =============================================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_tables = {
    "lineage_inventory": lineage_inventory,
    "case_multiplicity": case_multiplicity,
    "identifier_collisions": identifier_collisions,
    "gene_id_pilot": gene_id_pilot,
    "rna_adjudication_case_evidence": rna_adjudication_cases,
    "rna_adjudication_file_evidence": rna_adjudication_files,
    "rna_c_field_coverage": rna_c_field_coverage,
    "rna_c_file_availability": rna_c_file_availability,
    "rna_c_case_coverage": rna_c_case_coverage,
    "rna_d_identity": rna_d_identity,
    "rna_d_field_evidence": rna_d_field_evidence,
    "rna_d_case_summary": rna_d_case_summary,
    "rna_d_case_field_comparison": rna_d_case_field_comparison,
    "rna_e_file_diagnostics": rna_e_file_diagnostics,
    "rna_e_gene_coverage": rna_e_gene_coverage,
    "rna_e_source_attrition": rna_e_source_attrition,
}
for name, frame in output_tables.items():
    frame.to_csv(OUTPUT_PATHS[name], index=False)
    print(name, "→", project_relative_path(OUTPUT_PATHS[name]), "rows:", len(frame))

lineage_inventory → data/interim/metadata/800_preflight_a_candidate_lineage_inventory.csv rows: 61
case_multiplicity → data/interim/metadata/800_preflight_a_case_multiplicity.csv rows: 2706
identifier_collisions → data/interim/metadata/800_preflight_a_identifier_collisions.csv rows: 2706
gene_id_pilot → data/interim/metadata/800_preflight_a_gene_id_pilot.csv rows: 2389
rna_adjudication_case_evidence → data/interim/metadata/800_preflight_b_rna_case_evidence.csv rows: 288
rna_adjudication_file_evidence → data/interim/metadata/800_preflight_b_rna_file_evidence.csv rows: 649
rna_c_field_coverage → data/interim/metadata/800_preflight_c_rna_field_coverage.csv rows: 50
rna_c_file_availability → data/interim/metadata/800_preflight_c_rna_file_availability.csv rows: 649
rna_c_case_coverage → data/interim/metadata/800_preflight_c_rna_case_coverage.csv rows: 288
rna_d_identity → data/interim/metadata/800_preflight_d_rna_pilot_file_identity.csv rows: 29
rna_d_field_evidence → data/interim/metadata/

In [46]:
# =============================================================================
# Validate newly persisted technical inventories, not all upstream schemas
# =============================================================================

for name, frame in output_tables.items():
    reloaded = pd.read_csv(OUTPUT_PATHS[name], dtype="string")
    if len(reloaded) != len(frame) or list(reloaded.columns) != list(frame.columns):
        raise ValueError(f"New preflight output is inconsistent: {name}")

print("New technical inventory persistence: PASS")

New technical inventory persistence: PASS


In [47]:
# =============================================================================
# Write the non-frozen readiness record with provenance and output fingerprints
# =============================================================================

readiness_record = {
    "notebook": "800_external_bulk_cohorts",
    "stage": "PREFLIGHT_A_TO_E_SINGLE_PRIMARY_TECHNICAL_DIAGNOSTICS_ONLY",
    "environment": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
    "upstream_111_snapshot": audit_summary["source_fingerprints"],
    "inputs": input_provenance,
    "raw_registry_cptac3_present": cptac3_registered,
    "source_count_context": audit_summary["n_files_by_modality"],
    "program_gene_universe_n": len(shared_genes),
    "rna_case_multiplicity_class_counts": {
        str(row.rna_multiplicity_class): int(row.n_cases)
        for row in rna_class_summary.itertuples(index=False)
    },
    "rna_preflight_b_evidence_class_counts": {
        str(k): int(v) for k, v in
        rna_adjudication_cases["adjudication_evidence_class"]
        .value_counts(dropna=False).items()
    },
    "rna_preflight_b_case_count": len(rna_adjudication_cases),
    "rna_preflight_b_descriptor_discordance_cases": int(
        rna_adjudication_cases["source_descriptor_discordance"].sum()
    ),
    "rna_preflight_b_descriptor_incomplete_cases": int(
        rna_adjudication_cases["source_descriptor_incomplete"].sum()
    ),
    "rna_preflight_b_selection_performed": False,
    "rna_preflight_c_reviewed_files": len(rna_c_file_availability),
    "rna_preflight_c_reviewed_cases": len(rna_c_case_coverage),
    "rna_preflight_c_quantitative_qc_snapshot_files": int(
        rna_c_file_availability[
            "informative_quantitative_qc_metadata_probes"
        ].sum()
    ),
    "rna_preflight_c_selection_performed": False,
    "rna_preflight_d_pilot_cases": len(rna_d_case_summary),
    "rna_preflight_d_pilot_files": len(rna_d_identity),
    "rna_preflight_d_exact_linked_files": rna_d_exact_link_n,
    "rna_preflight_d_exact_link_status": rna_d_exact_link_status,
    "rna_preflight_d_prior_snapshot_sha256": (
        calculate_sha256(RNA_D_PRIOR_SNAPSHOT_PATH)
        if RNA_D_PRIOR_SNAPSHOT_PATH.is_file() else None
    ),
    "rna_preflight_d_case_fields_with_distinct_values": int(
        rna_d_case_field_comparison["candidate_values_differ"].sum()
    ),
    "rna_preflight_d_gdc_snapshot_sha256": calculate_sha256(
        RNA_D_SNAPSHOT_PATH
    ),
    "rna_preflight_d_pilot_selection_performed": False,
    "rna_preflight_d_full_649_expansion_performed": False,
    "rna_preflight_e_single_primary_files_scanned": len(
        rna_e_file_diagnostics
    ),
    "rna_preflight_e_source_contexts": len(rna_e_source_attrition),
    "rna_preflight_e_exact_frozen_ids_complete_files": int(
        rna_e_file_diagnostics["n_frozen_exact_missing"].eq(0).sum()
    ),
    "rna_preflight_e_other_same_case_rna_files": int(
        rna_e_file_diagnostics["n_other_rna_files_in_case"].gt(0).sum()
    ),
    "rna_preflight_e_eligible_cases_frozen": 0,
    "rna_preflight_e_selection_performed": False,
    "pilot_file_uuid": str(pilot_file_uuid),
    "pilot_star_counts_fields": pilot_header,
    "readiness": readiness_status,
    "cohort_frozen": False,
    "program_scores_computed": False,
    "external_replication_test_performed": False,
    "outputs": {
        name: {
            "path": project_relative_path(OUTPUT_PATHS[name]),
            "rows": len(output_tables[name]),
            "sha256": calculate_sha256(OUTPUT_PATHS[name]),
        }
        for name in output_tables
    },
}
with OUTPUT_PATHS["readiness"].open("w", encoding="utf-8") as handle:
    json.dump(readiness_record, handle, indent=2, ensure_ascii=False)
    handle.write("\n")
with OUTPUT_PATHS["readiness"].open("r", encoding="utf-8") as handle:
    persisted_readiness = json.load(handle)
if persisted_readiness["readiness"] != readiness_status:
    raise ValueError("Persisted readiness differs from the in-memory record")
print("Readiness saved:", project_relative_path(OUTPUT_PATHS["readiness"]))

Readiness saved: data/interim/metadata/800_preflight_a_readiness.json


## Notebook 800 checkpoint — PREFLIGHT-A–D reviewed, E inventory pending; cohort NOT frozen

**Technical and scientific boundary.** Notebook 800 completed its outcome-blind acquisition and external-input eligibility diagnostics through the **PREFLIGHT-D v2 pilot** (41/41 code cells executed, no exceptions in the recorded run). It reuses, without recomputing, Notebook 111 acquisition integrity results and checks the frozen Phase 2/4 inputs. **No case/file adjudication, new sample-ancestry promotion, program scoring, biological replication, registry promotion or cohort freeze has occurred.**

### PREFLIGHT-A–C — reused technical findings

- **A:** 1,370 provisionally mapped RNA primary-tumor cases: 1,082 with one file/sample/aliquot, 276 with multiple samples and 12 with multiple aliquots. The single STAR Counts pilot matched 2,389 frozen genes; full cohort compatibility remains open. Zero lexical ID collisions with the frozen TCGA mapping do **not** prove independent donors.
- **B:** 288 multiplicity cases comprising 649 files. The audited generic provider descriptors were complete and concordant within the queried fields but could not justify choosing any candidate.
- **C:** the original acquisition-time GDC RNA metadata snapshot was fingerprint-matched to Notebook 111. The prespecified specific collection-time, spatial-location and quantitative-QC probes yielded no informative values in that **original projection**. This was not a provider-wide negative result.

### PREFLIGHT-D — first pilot blocker and v2 resolution

The deterministic bounded pilot reviewed **12 cases and 29 existing RNA file UUIDs** (three cases each from source strata uterus, kidney, larynx and pancreas). The first live `/files` query confirmed 29/29 direct file–aliquot links but recovered **0/29** fully identifiable exact case/sample/aliquot nested paths: that first provenance-availability result was **BLOCKED**, not informative.

**D v2 resolved the technical linkage limitation.** The updated `/files` plus exact-case `/cases` queries recovered **29/29 exact, previously resolved case/sample/aliquot triplets** while preserving the direct file–aliquot identity. The original v1 snapshot was retained; the separate v2 snapshot is fingerprinted and was not used to reassign ambiguous sample ancestry.

**Observed v2 evidence:** 29/29 files had at least one informative general anatomical descriptor and at least one collection/procurement descriptor in the new projection. Other probed fields showed partial coverage (e.g., laterality and time between excision and freezing). **Zero of the compared case-field combinations showed distinct informative values between candidate files within the same case**; therefore these newly recovered descriptors offered no demonstrated *within-case sample-selection rule* in this pilot. No requested file-level quantitative QC or read-group QC records were present in the 29 pilot file responses. Absence from these responses does **not** establish that QC is unavailable in other sources or processed STAR payloads. The pilot is not evidence that all 649 files lack discriminating fields.

### Closure boundary and next decision

**Accept D v2 as a technically successful, bounded provenance-availability pilot, not as case adjudication.** The 288 multiplicity cases remain unresolved. The pilot supplies no demonstrated benefit from automatically expanding the same metadata query to all 649 files; any expansion would need a new, prospective rationale. The next decision is a conservative, lineage-aware eligibility and technical QC plan based on existing uniquely resolved files, independent QC provenance if available, and explicit handling of non-independent multiple samples. Do not automatically promote the 1,082 structurally simple cases: technical QC, tumor context, cohort overlap, gene coverage, preprocessing compatibility and donor independence must still be audited.

The one-observation-per-case policy, lineage crosswalk, donor independence, full transcriptomic QC, program-replication estimand and matched nulls are **NOT FROZEN**. `biological_validation_authorized=false`; no external program-replication inference is permitted.

### PREFLIGHT-E implementation — NOT YET EXECUTED

The new E-stage code examines only the 1,082 single-provisional-primary-file cases already defined by 800-A, scans downloaded STAR `gene_id` and `unstranded` counts for technical diagnostics, quantifies exact coverage of the frozen 2,389 gene IDs (without version rescue), flags other same-case RNA files from Notebook 111 and reports structural attrition by source site/disease labels. It does not adjudicate the 288 multiplicity cases or determine QC thresholds, lineage harmonization, donor independence, eligibility, normalization or biological replication. Interpretation and any cohort freeze require review of the actual E output tables after local Run All.